In [ ]:
# 🔍 DIAGNOSTIC : Pourquoi le ML ne fonctionne pas ?
print("🔍 Diagnostic des données pour le Machine Learning")
print("=" * 60)

# Vérifier les variables de statut
print(f"📊 Variables de statut:")
print(f"   • env_data_found: {env_data_found}")
print(f"   • ml_ready: {ml_ready}")
print(f"   • has_grid: {has_grid}")

# Vérifier les DataFrames disponibles
print(f"\n📋 DataFrames disponibles:")
if 'vis_data' in locals():
    print(f"   • vis_data: {vis_data.shape[0]:,} lignes, {vis_data.shape[1]} colonnes")
    print(f"     Colonnes: {list(vis_data.columns)}")
    
    # Vérifier les colonnes environnementales clés
    env_cols = ['temperature_param_mean', 'depth_param_mean', 'salinity_param_mean', 
                'primary_production_param_mean', 'dissolved_oxygen_param_mean']
    
    print(f"\n🌊 Données environnementales:")
    for col in env_cols:
        if col in vis_data.columns:
            non_null = vis_data[col].notna().sum()
            total = len(vis_data)
            pct = (non_null/total*100) if total > 0 else 0
            print(f"   • {col}: {non_null:,}/{total:,} ({pct:.1f}%) données valides")
        else:
            print(f"   • {col}: ❌ MANQUANT")
    
    # Vérifier les données complètes (toutes colonnes env)
    if all(col in vis_data.columns for col in env_cols):
        complete_rows = vis_data[env_cols].dropna()
        print(f"\n✅ Lignes avec TOUTES les données environnementales: {len(complete_rows):,}/{len(vis_data):,}")
        if len(complete_rows) < 100:
            print(f"⚠️  PROBLÈME IDENTIFIÉ: Seulement {len(complete_rows)} lignes complètes (minimum peut-être 100+)")
    
else:
    print(f"   • vis_data: ❌ NON DISPONIBLE")

if 'df_grid' in locals():
    print(f"   • df_grid: {df_grid.shape[0]:,} lignes, {df_grid.shape[1]} colonnes")
else:
    print(f"   • df_grid: ❌ NON DISPONIBLE")

# Proposer une solution
print(f"\n💡 Solution suggérée:")
if 'vis_data' in locals() and len(vis_data) > 0:
    env_cols_available = [col for col in ['temperature_param_mean', 'depth_param_mean', 'salinity_param_mean'] 
                         if col in vis_data.columns and vis_data[col].notna().sum() > 50]
    if len(env_cols_available) >= 2:
        print(f"   ✅ Forcer ml_ready = True (suffisamment de données)")
        print(f"   📊 Colonnes utilisables: {env_cols_available}")
        # Forcer la variable
        ml_ready = True
        print(f"   🔧 ml_ready forcé à: {ml_ready}")
    else:
        print(f"   ❌ Pas assez de colonnes environnementales avec données")
else:
    print(f"   ❌ Aucune donnée disponible pour le ML")

In [1]:
# Définir la fonction get_cell_center_coords pour éviter les conflits
def get_cell_center_coords(grid_index, min_val, max_val, grid_size):
    """Obtenir le centre d'une cellule de grille (version 4 paramètres)"""
    if pd.isna(grid_index):
        return None
    cell_size = (max_val - min_val) / grid_size
    return min_val + (grid_index + 0.5) * cell_size

print("🔧 Fonction get_cell_center_coords définie globalement")

🔧 Fonction get_cell_center_coords définie globalement


# 🌍 Carte Interactive des Espèces Marines - Grille 50×50

Ce notebook crée une carte interactive du monde divisée en grille 50×50 pour analyser la distribution des espèces marines. Chaque cellule de la grille affiche des informations moyennes sur :

- 🌡️ **Température moyenne**
- 🌊 **Profondeur moyenne** 
- 📊 **Probabilité d'occurrence moyenne**
- 🧂 **Salinité moyenne**
- 🐠 **Nombre d'espèces observées**

## Fonctionnalités
- Visualisation interactive avec Plotly
- Grille mondiale 50×50 (3600 cellules)
- Informations détaillées au survol
- Échantillon de 500,000 observations
- Calculs d'agrégation par zone géographique

In [2]:
# Import des librairies nécessaires
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import warnings
warnings.filterwarnings('ignore')

# Configuration pour Plotly
px.defaults.template = "plotly_white"
px.defaults.color_continuous_scale = "Viridis"

print("📚 Librairies importées avec succès!")
print("🌊 Prêt pour l'analyse des données marines...")

📚 Librairies importées avec succès!
🌊 Prêt pour l'analyse des données marines...


In [3]:
# Définitions des fonctions de grille (nécessaires pour l'analyse des espèces)
def grid_index(coord, min_val, max_val, grid_size):
    """Calculer l'index de grille pour une coordonnée"""
    if pd.isna(coord):
        return None
    
    # Normaliser la coordonnée entre 0 et 1
    normalized = (coord - min_val) / (max_val - min_val)
    
    # Calculer l'index de la grille (0 à grid_size-1)
    index = int(normalized * grid_size)
    index = max(0, min(index, grid_size - 1))
    
    return index

def get_cell_center_coords(grid_index, min_val, max_val, grid_size):
    """Obtenir le centre d'une cellule de grille (version 4 paramètres)"""
    if pd.isna(grid_index):
        return None
    cell_size = (max_val - min_val) / grid_size
    return min_val + (grid_index + 0.5) * cell_size

def get_cell_center(grid_index, min_val, max_val, grid_size):
    """Obtenir le centre d'une cellule de grille (version alternative)"""
    cell_size = (max_val - min_val) / grid_size
    return min_val + (grid_index + 0.5) * cell_size

print("✅ Toutes les fonctions de grille définies avec succès")
print("🔧 Fonctions disponibles: grid_index, get_cell_center_coords, get_cell_center")

✅ Toutes les fonctions de grille définies avec succès
🔧 Fonctions disponibles: grid_index, get_cell_center_coords, get_cell_center


In [4]:
# Chargement et échantillonnage des données avec approche par chunks
print("📁 Chargement du fichier marine_species_expanded.csv...")
print("⚠️ Fichier volumineux détecté - utilisation de l'approche par chunks")

# Paramètres d'échantillonnage
sample_size = 500000
chunk_size = 10000  # Lire 10k lignes à la fois
np.random.seed(42)  # Pour la reproductibilité

# Fonction pour échantillonner avec chunks
def sample_large_csv(filepath, sample_size, chunk_size):
    """
    Échantillonne un gros fichier CSV en utilisant des chunks
    """
    print(f"🔄 Lecture par chunks de {chunk_size:,} lignes...")
    
    # Première passe : compter le nombre total de lignes valides
    total_valid_rows = 0
    chunk_count = 0
    
    print("📊 Première passe : comptage des lignes valides...")
    for chunk in pd.read_csv(filepath, chunksize=chunk_size):
        chunk_count += 1
        # Compter les lignes avec coordonnées valides
        valid_chunk = chunk.dropna(subset=['center_lat', 'center_long'])
        total_valid_rows += len(valid_chunk)
        
        if chunk_count % 50 == 0:  # Affichage du progrès
            print(f"   Chunks traités: {chunk_count}, Lignes valides trouvées: {total_valid_rows:,}")
    
    print(f"✅ Total de lignes valides: {total_valid_rows:,}")
    
    # Calculer la probabilité d'échantillonnage
    if total_valid_rows <= sample_size:
        sampling_prob = 1.0
        print(f"📝 Toutes les lignes valides seront conservées")
    else:
        sampling_prob = sample_size / total_valid_rows
        print(f"📝 Probabilité d'échantillonnage: {sampling_prob:.4f}")
    
    # Deuxième passe : échantillonnage
    sampled_data = []
    chunk_count = 0
    rows_sampled = 0
    
    print(f"🎯 Deuxième passe : échantillonnage de {sample_size:,} observations...")
    
    for chunk in pd.read_csv(filepath, chunksize=chunk_size):
        chunk_count += 1
        
        # Filtrer les lignes valides
        valid_chunk = chunk.dropna(subset=['center_lat', 'center_long'])
        
        if len(valid_chunk) > 0:
            # Échantillonner ce chunk
            if sampling_prob >= 1.0:
                sample_chunk = valid_chunk
            else:
                # Échantillonnage aléatoire
                n_sample = int(len(valid_chunk) * sampling_prob)
                if n_sample > 0:
                    sample_chunk = valid_chunk.sample(n=n_sample, random_state=42+chunk_count)
                else:
                    # Au moins une ligne si le chunk contient des données
                    if np.random.random() < sampling_prob:
                        sample_chunk = valid_chunk.sample(n=1, random_state=42+chunk_count)
                    else:
                        continue
            
            sampled_data.append(sample_chunk)
            rows_sampled += len(sample_chunk)
            
            # Affichage du progrès
            if chunk_count % 50 == 0:
                print(f"   Chunks: {chunk_count}, Échantillonnées: {rows_sampled:,}")
            
            # Arrêter si on a assez d'échantillons
            if rows_sampled >= sample_size:
                print(f"🎯 Objectif atteint : {rows_sampled:,} lignes échantillonnées")
                break
    
    # Combiner tous les chunks échantillonnés
    if sampled_data:
        df_sample = pd.concat(sampled_data, ignore_index=True)
        # Limiter à la taille exacte demandée
        if len(df_sample) > sample_size:
            df_sample = df_sample.sample(n=sample_size, random_state=42)
        return df_sample
    else:
        print("❌ Aucune donnée échantillonnée")
        return pd.DataFrame()

# Échantillonner le fichier
try:
    df_sample = sample_large_csv('Scrapping/marine_species_expanded.csv', sample_size, chunk_size)
    
    if len(df_sample) > 0:
        print(f"🎯 Échantillon final: {len(df_sample):,} observations")
        print(f"🗺️ Plage des latitudes: {df_sample['center_lat'].min():.2f}° à {df_sample['center_lat'].max():.2f}°")
        print(f"🗺️ Plage des longitudes: {df_sample['center_long'].min():.2f}° à {df_sample['center_long'].max():.2f}°")
        
        # Affichage des colonnes disponibles
        print(f"\n📋 Colonnes disponibles ({len(df_sample.columns)}):")
        for i, col in enumerate(df_sample.columns, 1):
            print(f"{i:2d}. {col}")
        
        # Statistiques de base
        print(f"\n📊 Statistiques de base:")
        print(f"   • Espèces uniques: {df_sample['species'].nunique():,}")
        print(f"   • Genres uniques: {df_sample['genus'].nunique():,}")
        
        if 'temperature_param_mean' in df_sample.columns:
            temp_valid = df_sample['temperature_param_mean'].dropna()
            if len(temp_valid) > 0:
                print(f"   • Température: {temp_valid.min():.1f}°C à {temp_valid.max():.1f}°C")
            else:
                print(f"   • Température: Aucune donnée valide")
        
        if 'depth_param_mean' in df_sample.columns:
            depth_valid = df_sample['depth_param_mean'].dropna()
            if len(depth_valid) > 0:
                print(f"   • Profondeur: {depth_valid.min():.0f}m à {depth_valid.max():.0f}m")
            else:
                print(f"   • Profondeur: Aucune donnée valide")
        
        # Vérifier la qualité des données
        missing_coords = df_sample[['center_lat', 'center_long']].isnull().any(axis=1).sum()
        print(f"\n🔍 Qualité des données:")
        print(f"   • Coordonnées manquantes: {missing_coords} lignes")
        print(f"   • Coordonnées valides: {len(df_sample) - missing_coords} lignes")
        
    else:
        print("❌ Échec de l'échantillonnage - fichier vide ou inaccessible")
        # Créer un DataFrame vide avec les colonnes essentielles
        df_sample = pd.DataFrame(columns=['center_lat', 'center_long', 'species', 'genus'])
        
except Exception as e:
    print(f"❌ Erreur lors du chargement: {str(e)}")
    print("🔄 Tentative avec un fichier plus petit ou vérifiez le chemin du fichier")
    # Créer un DataFrame vide pour éviter les erreurs dans les cellules suivantes
    df_sample = pd.DataFrame(columns=['center_lat', 'center_long', 'species', 'genus'])

📁 Chargement du fichier marine_species_expanded.csv...
⚠️ Fichier volumineux détecté - utilisation de l'approche par chunks
🔄 Lecture par chunks de 10,000 lignes...
📊 Première passe : comptage des lignes valides...
   Chunks traités: 50, Lignes valides trouvées: 499,957
   Chunks traités: 50, Lignes valides trouvées: 499,957
   Chunks traités: 100, Lignes valides trouvées: 999,944
   Chunks traités: 100, Lignes valides trouvées: 999,944
   Chunks traités: 150, Lignes valides trouvées: 1,499,887
   Chunks traités: 150, Lignes valides trouvées: 1,499,887
   Chunks traités: 200, Lignes valides trouvées: 1,999,878
   Chunks traités: 200, Lignes valides trouvées: 1,999,878
   Chunks traités: 250, Lignes valides trouvées: 2,499,877
   Chunks traités: 250, Lignes valides trouvées: 2,499,877
   Chunks traités: 300, Lignes valides trouvées: 2,999,877
   Chunks traités: 300, Lignes valides trouvées: 2,999,877
   Chunks traités: 350, Lignes valides trouvées: 3,499,877
   Chunks traités: 350, Lign

In [5]:
# Définition de la grille mondiale 50×50
print("🌐 Création de la grille mondiale 50×50...")

# Vérifier que nous avons des données valides
if len(df_sample) == 0:
    print("❌ Aucune donnée disponible pour créer la grille")
    print("🔄 Veuillez vérifier le fichier CSV et relancer l'échantillonnage")
else:
    print(f"✅ Données disponibles: {len(df_sample):,} observations")
    
    # Paramètres de la grille
    grid_size = 50
    lat_min, lat_max = -90, 90
    lon_min, lon_max = -180, 180
    
    # Calcul des intervalles
    lat_step = (lat_max - lat_min) / grid_size  # 180/50 = 3.6° par cellule
    lon_step = (lon_max - lon_min) / grid_size  # 360/50 = 7.2° par cellule
    
    print(f"📐 Taille de chaque cellule: {lat_step:.2f}° lat × {lon_step:.2f}° lon")
    print(f"📦 Nombre total de cellules: {grid_size * grid_size:,}")
    
    # Création des limites de grille
    lat_edges = np.linspace(lat_min, lat_max, grid_size + 1)
    lon_edges = np.linspace(lon_min, lon_max, grid_size + 1)
    
    # Assignation des observations aux cellules de grille
    def assign_to_grid(lat, lon):
        """Assigne une coordonnée lat/lon à un index de grille"""
        if pd.isna(lat) or pd.isna(lon):
            return np.nan, np.nan
        
        # Trouver l'index de grille pour latitude
        lat_idx = np.clip(int((lat - lat_min) / lat_step), 0, grid_size - 1)
        
        # Trouver l'index de grille pour longitude
        lon_idx = np.clip(int((lon - lon_min) / lon_step), 0, grid_size - 1)
        
        return lat_idx, lon_idx
    
    # Appliquer l'assignation à toutes les observations
    print("⚙️ Attribution des observations aux cellules...")
    df_sample[['lat_grid_idx', 'lon_grid_idx']] = df_sample.apply(
        lambda row: assign_to_grid(row['center_lat'], row['center_long']), 
        axis=1, result_type='expand'
    )
    
    # Filtrer les observations avec des indices valides
    df_grid = df_sample.dropna(subset=['lat_grid_idx', 'lon_grid_idx']).copy()
    df_grid['lat_grid_idx'] = df_grid['lat_grid_idx'].astype(int)
    df_grid['lon_grid_idx'] = df_grid['lon_grid_idx'].astype(int)
    
    print(f"✅ {len(df_grid):,} observations assignées aux cellules de grille")
    
    if len(df_grid) == 0:
        print("⚠️ Aucune observation n'a pu être assignée à la grille")
        print("🔍 Vérifiez la qualité des coordonnées dans les données")
    
    # Calculer les centres des cellules pour l'affichage
    def get_cell_center(lat_idx, lon_idx):
        """Calcule le centre d'une cellule de grille"""
        lat_center = lat_min + (lat_idx + 0.5) * lat_step
        lon_center = lon_min + (lon_idx + 0.5) * lon_step
        return lat_center, lon_center
    
    def get_cell_bounds(lat_idx, lon_idx):
        """Calcule les limites d'une cellule de grille"""
        lat_south = lat_min + lat_idx * lat_step
        lat_north = lat_min + (lat_idx + 1) * lat_step
        lon_west = lon_min + lon_idx * lon_step
        lon_east = lon_min + (lon_idx + 1) * lon_step
        return lat_south, lat_north, lon_west, lon_east

🌐 Création de la grille mondiale 50×50...
✅ Données disponibles: 499,281 observations
📐 Taille de chaque cellule: 3.60° lat × 7.20° lon
📦 Nombre total de cellules: 2,500
⚙️ Attribution des observations aux cellules...
✅ 499,281 observations assignées aux cellules de grille
✅ 499,281 observations assignées aux cellules de grille


In [6]:
# Agrégation des observations par cellule de grille
print("📊 Agrégation des données par cellule...")

# Vérifier que nous avons des données de grille
if 'df_grid' not in locals() or len(df_grid) == 0:
    print("❌ Aucune donnée de grille disponible pour l'agrégation")
    print("🔄 Veuillez vérifier les étapes précédentes")
    # Créer des variables vides pour éviter les erreurs
    grid_summary = pd.DataFrame()
    available_cols = []
else:
    # Colonnes numériques pour l'agrégation
    numeric_cols = [
        'temperature_param_mean', 'depth_param_mean', 'salinity_param_mean',
        'primary_production_param_mean', 'dissolved_oxygen_param_mean',
        'distance_to_land_param_mean', 'overall_probability'
    ]
    
    # Vérifier quelles colonnes existent dans le dataset
    available_cols = [col for col in numeric_cols if col in df_grid.columns]
    print(f"📋 Colonnes disponibles pour l'agrégation: {available_cols}")
    
    if len(available_cols) == 0:
        print("⚠️ Aucune colonne numérique trouvée pour l'agrégation")
        print("📋 Colonnes disponibles dans les données:")
        print(list(df_grid.columns))
    
    # Agrégation par cellule de grille
    agg_functions = {
        'species': 'nunique',  # Nombre d'espèces uniques
        'genus': 'nunique',    # Nombre de genres uniques
        'center_lat': 'count'  # Nombre total d'observations
    }
    
    # Ajouter les fonctions d'agrégation pour les colonnes numériques
    for col in available_cols:
        agg_functions[col] = 'mean'
    
    # Effectuer l'agrégation
    try:
        grid_summary = df_grid.groupby(['lat_grid_idx', 'lon_grid_idx']).agg(agg_functions).reset_index()
        
        # Renommer les colonnes pour plus de clarté
        column_rename = {
            'species': 'species_count',
            'genus': 'genus_count', 
            'center_lat': 'observation_count'
        }
        grid_summary.rename(columns=column_rename, inplace=True)
        
        # Calculer les centres et limites des cellules
        print("🎯 Calcul des centres et limites des cellules...")
        grid_summary[['cell_center_lat', 'cell_center_lon']] = grid_summary.apply(
            lambda row: get_cell_center(row['lat_grid_idx'], row['lon_grid_idx']), 
            axis=1, result_type='expand'
        )
        
        # Ajouter les limites des cellules
        bounds_data = grid_summary.apply(
            lambda row: get_cell_bounds(row['lat_grid_idx'], row['lon_grid_idx']), 
            axis=1, result_type='expand'
        )
        grid_summary[['lat_south', 'lat_north', 'lon_west', 'lon_east']] = bounds_data
        
        print(f"✅ Grille complétée avec {len(grid_summary):,} cellules contenant des données")
        print(f"📈 Moyenne d'observations par cellule: {grid_summary['observation_count'].mean():.1f}")
        print(f"📊 Cellule la plus dense: {grid_summary['observation_count'].max():,} observations")
        
        # Affichage d'un échantillon
        print("\n🔍 Aperçu des données de grille:")
        print(grid_summary.head())
        
    except Exception as e:
        print(f"❌ Erreur lors de l'agrégation: {str(e)}")
        grid_summary = pd.DataFrame()
        available_cols = []

📊 Agrégation des données par cellule...
📋 Colonnes disponibles pour l'agrégation: ['temperature_param_mean', 'depth_param_mean', 'salinity_param_mean', 'primary_production_param_mean', 'dissolved_oxygen_param_mean', 'distance_to_land_param_mean', 'overall_probability']
🎯 Calcul des centres et limites des cellules...
✅ Grille complétée avec 1,906 cellules contenant des données
📈 Moyenne d'observations par cellule: 262.0
📊 Cellule la plus dense: 1,772 observations

🔍 Aperçu des données de grille:
   lat_grid_idx  lon_grid_idx  species_count  genus_count  observation_count  \
0             3             0             31           26                 75   
1             3             1             36           27                 92   
2             3             2             31           24                 67   
3             3             3             38           28                 79   
4             3             4             24           18                 36   

   temperature_para

In [7]:
# Création de la carte interactive principale unique
print("🗺️ Création de la carte interactive unique avec toutes les informations...")

# Vérifier que nous avons des données de grille
if 'grid_summary' not in locals() or len(grid_summary) == 0:
    print("❌ Aucune donnée de grille disponible pour la visualisation")
    print("🔄 Veuillez vérifier les étapes précédentes d'agrégation")
    print("\n💡 Suggestions:")
    print("   1. Vérifiez que le fichier CSV existe et est accessible")
    print("   2. Assurez-vous que les colonnes 'center_lat' et 'center_long' existent")
    print("   3. Vérifiez que l'échantillonnage a fonctionné correctement")
else:
    # Préparer les données pour la visualisation
    vis_data = grid_summary.copy()
    
    print(f"📊 Données disponibles: {len(vis_data)} cellules")
    
    # Choisir la variable principale pour la couleur (nombre d'observations par défaut)
    # mais on peut utiliser température si disponible
    if 'temperature_param_mean' in vis_data.columns and vis_data['temperature_param_mean'].notna().sum() > 0:
        color_var = 'temperature_param_mean'
        color_label = 'Température Moyenne (°C)'
        color_scale = 'RdYlBu_r'
    elif 'depth_param_mean' in vis_data.columns and vis_data['depth_param_mean'].notna().sum() > 0:
        color_var = 'depth_param_mean'
        color_label = 'Profondeur Moyenne (m)'
        color_scale = 'Blues_r'
    else:
        color_var = 'observation_count'
        color_label = 'Nombre d\'Observations'
        color_scale = 'Viridis'
    
    print(f"🎨 Variable de couleur sélectionnée: {color_var}")
    
    # Créer le texte de hover complet avec TOUTES les informations disponibles
    hover_text = []
    for _, row in vis_data.iterrows():
        text_parts = [
            f"<b>🌍 Cellule de Grille ({row['lat_grid_idx']}, {row['lon_grid_idx']})</b>",
            f"<b>📍 Position:</b>",
            f"   • Centre: {row['cell_center_lat']:.2f}°N, {row['cell_center_lon']:.2f}°E",
            f"   • Zone: {row['lat_south']:.2f}° à {row['lat_north']:.2f}° lat",
            f"   •       {row['lon_west']:.2f}° à {row['lon_east']:.2f}° lon",
            f"<b>🔢 Observations Biologiques:</b>",
            f"   • Total observations: {row['observation_count']:,}",
            f"   • Espèces uniques: {row['species_count']:,}",
            f"   • Genres uniques: {row['genus_count']:,}"
        ]
        
        # Section des données environnementales
        env_data_found = False
        env_parts = ["<b>🌊 Données Environnementales:</b>"]
        
        if 'temperature_param_mean' in row and not pd.isna(row['temperature_param_mean']):
            env_parts.append(f"   • 🌡️ Température: {row['temperature_param_mean']:.1f}°C")
            env_data_found = True
        
        if 'depth_param_mean' in row and not pd.isna(row['depth_param_mean']):
            env_parts.append(f"   • 🌊 Profondeur: {row['depth_param_mean']:.0f}m")
            env_data_found = True
        
        if 'salinity_param_mean' in row and not pd.isna(row['salinity_param_mean']):
            env_parts.append(f"   • 🧂 Salinité: {row['salinity_param_mean']:.1f} PSU")
            env_data_found = True
        
        if 'primary_production_param_mean' in row and not pd.isna(row['primary_production_param_mean']):
            env_parts.append(f"   • 🌱 Production primaire: {row['primary_production_param_mean']:.2f}")
            env_data_found = True
        
        if 'dissolved_oxygen_param_mean' in row and not pd.isna(row['dissolved_oxygen_param_mean']):
            env_parts.append(f"   • 💨 Oxygène dissous: {row['dissolved_oxygen_param_mean']:.2f}")
            env_data_found = True
        
        if 'distance_to_land_param_mean' in row and not pd.isna(row['distance_to_land_param_mean']):
            env_parts.append(f"   • 🏝️ Distance à la terre: {row['distance_to_land_param_mean']:.0f}km")
            env_data_found = True
        
        if 'overall_probability' in row and not pd.isna(row['overall_probability']):
            env_parts.append(f"   • 📊 Probabilité occurrence: {row['overall_probability']:.3f}")
            env_data_found = True
        
        if env_data_found:
            text_parts.extend(env_parts)
        else:
            text_parts.append("<b>🌊 Données Environnementales:</b> Non disponibles")
        
        # Ajouter la zone climatique si disponible
        if 'zone' in row and not pd.isna(row['zone']):
            text_parts.append(f"<b>🌡️ Zone climatique:</b> {row['zone']}")
        
        hover_text.append("<br>".join(text_parts))
    
    vis_data['hover_text'] = hover_text
    
    try:
        # Créer la carte unique avec Plotly
        fig = px.scatter_mapbox(
            vis_data,
            lat='cell_center_lat',
            lon='cell_center_lon',
            color=color_var,
            size='observation_count',
            hover_name=None,  # Pas de nom par défaut
            color_continuous_scale=color_scale,
            size_max=20,  # Augmenté pour une meilleure visibilité
            zoom=1.2,
            height=900,  # Plus haute pour une meilleure visualisation
            title=f"🌍 Carte Interactive Mondiale des Espèces Marines - Grille 50×50<br><sub>🎨 Couleur: {color_label} | 📏 Taille: Nombre d'observations | 📊 {len(vis_data)} cellules avec données</sub>"
        )
        
        # Configuration avancée de la carte
        fig.update_layout(
            mapbox_style="open-street-map",
            margin={"r":10,"t":80,"l":10,"b":10},
            title_font_size=18,
            title_x=0.5,  # Centrer le titre
            coloraxis_colorbar=dict(
                title=dict(text=color_label, side="right"),
                len=0.8,  # Plus long
                thickness=15
            ),
            font=dict(size=12)
        )
        
        # Configuration du hover avec notre texte personnalisé
        fig.update_traces(
            hovertemplate='%{hovertext}<extra></extra>',
            hovertext=vis_data['hover_text'],
            marker=dict(
                opacity=0.8  # Légère transparence
            )
        )
        
        print(f"✅ Carte unique créée avec {len(vis_data)} cellules")
        print(f"🎨 Couleur basée sur: {color_label}")
        print(f"📏 Taille basée on: Nombre d'observations")
        print("🖱️ Survolez les points pour voir toutes les informations détaillées de chaque cellule!")
        print("")
        print("📋 Informations disponibles dans le hover:")
        print("   • Position géographique et limites de la cellule")
        print("   • Nombre d'observations, d'espèces et de genres")
        print("   • Variables environnementales (température, profondeur, salinité, etc.)")
        print("   • Zone climatique")
        
        # Afficher la carte
        fig.show()
        
        # Statistiques finales
        print(f"\n📊 RÉSUMÉ DE LA CARTE:")
        print(f"   🗺️ Cellules visualisées: {len(vis_data):,}")
        print(f"   🔢 Total observations: {vis_data['observation_count'].sum():,}")
        print(f"   🐠 Espèces totales: {vis_data['species_count'].sum():,}")
        print(f"   🧬 Genres totaux: {vis_data['genus_count'].sum():,}")
        
        if color_var != 'observation_count':
            color_stats = vis_data[color_var].dropna()
            if len(color_stats) > 0:
                print(f"   🎨 {color_label}: min={color_stats.min():.2f}, max={color_stats.max():.2f}, moyenne={color_stats.mean():.2f}")
        
    except Exception as e:
        print(f"❌ Erreur lors de la création de la carte: {str(e)}")
        print("💡 Vérifiez votre connexion internet pour l'affichage des cartes")

🗺️ Création de la carte interactive unique avec toutes les informations...
📊 Données disponibles: 1906 cellules
🎨 Variable de couleur sélectionnée: temperature_param_mean
✅ Carte unique créée avec 1906 cellules
🎨 Couleur basée sur: Température Moyenne (°C)
📏 Taille basée on: Nombre d'observations
🖱️ Survolez les points pour voir toutes les informations détaillées de chaque cellule!

📋 Informations disponibles dans le hover:
   • Position géographique et limites de la cellule
   • Nombre d'observations, d'espèces et de genres
   • Variables environnementales (température, profondeur, salinité, etc.)
   • Zone climatique
✅ Carte unique créée avec 1906 cellules
🎨 Couleur basée sur: Température Moyenne (°C)
📏 Taille basée on: Nombre d'observations
🖱️ Survolez les points pour voir toutes les informations détaillées de chaque cellule!

📋 Informations disponibles dans le hover:
   • Position géographique et limites de la cellule
   • Nombre d'observations, d'espèces et de genres
   • Variable


📊 RÉSUMÉ DE LA CARTE:
   🗺️ Cellules visualisées: 1,906
   🔢 Total observations: 499,281
   🐠 Espèces totales: 193,123
   🧬 Genres totaux: 155,715
   🎨 Température Moyenne (°C): min=-0.55, max=25.69, moyenne=15.37


In [8]:
# Création d'un planisphère avec rectangles colorés - Visualisation claire de la grille
print("🗺️ Création du planisphère avec rectangles colorés...")

# Vérifier que nous avons des données de grille
if 'grid_summary' not in locals() or len(grid_summary) == 0:
    print("❌ Aucune donnée de grille disponible pour la visualisation")
else:
    # Préparer les données pour la visualisation rectangulaire
    vis_data = grid_summary.copy()
    
    print(f"📊 Création du planisphère avec {len(vis_data)} cellules")
    
    # Choisir la variable pour la couleur
    if 'temperature_param_mean' in vis_data.columns and vis_data['temperature_param_mean'].notna().sum() > 0:
        color_var = 'temperature_param_mean'
        color_label = 'Température Moyenne (°C)'
        color_scale = 'RdYlGn_r'  # Rouge (chaud) à Vert (froid) - très contrasté
    elif 'depth_param_mean' in vis_data.columns and vis_data['depth_param_mean'].notna().sum() > 0:
        color_var = 'depth_param_mean'
        color_label = 'Profondeur Moyenne (m)'
        color_scale = 'plasma'  # Violet à Jaune - très visible
    else:
        color_var = 'observation_count'
        color_label = 'Nombre d\'Observations'
        color_scale = 'turbo'  # Multicolore très contrasté
    
    print(f"🎨 Variable de couleur: {color_label}")
    
    # Créer une matrice 50x50 pour la heatmap
    import numpy as np
    
    # Initialiser la matrice avec des NaN
    heatmap_matrix = np.full((50, 50), np.nan)
    hover_matrix = np.full((50, 50), '', dtype=object)
    
    # Remplir la matrice avec les données
    for _, row in vis_data.iterrows():
        lat_idx = int(row['lat_grid_idx'])
        lon_idx = int(row['lon_grid_idx'])
        
        # Valeur pour la couleur
        if not pd.isna(row[color_var]):
            heatmap_matrix[49-lat_idx, lon_idx] = row[color_var]  # Inverser lat pour affichage correct
        
        # Texte de hover
        hover_parts = [
            f"Cellule ({lat_idx}, {lon_idx})",
            f"📍 {row['lat_south']:.1f}° à {row['lat_north']:.1f}°N",
            f"    {row['lon_west']:.1f}° à {row['lon_east']:.1f}°E",
            f"📊 {row['observation_count']:,} obs.",
            f"🐠 {row['species_count']:,} espèces"
        ]
        
        if 'temperature_param_mean' in row and not pd.isna(row['temperature_param_mean']):
            hover_parts.append(f"🌡️ {row['temperature_param_mean']:.1f}°C")
        if 'depth_param_mean' in row and not pd.isna(row['depth_param_mean']):
            hover_parts.append(f"🌊 {row['depth_param_mean']:.0f}m")
        
        hover_matrix[49-lat_idx, lon_idx] = "<br>".join(hover_parts)
    
    # Créer les axes pour les labels
    lat_labels = [f"{90-i*3.6:.1f}°N" for i in range(0, 50, 5)]  # Tous les 5 indices
    lon_labels = [f"{-180+i*7.2:.1f}°E" for i in range(0, 50, 5)]
    
    # Créer la heatmap
    fig = go.Figure(data=go.Heatmap(
        z=heatmap_matrix,
        text=hover_matrix,
        hovertemplate='%{text}<extra></extra>',
        colorscale=color_scale,
        showscale=True,
        colorbar=dict(
            title=dict(text=color_label, side="right"),
            len=0.8,
            thickness=20
        )
    ))
    
    # Configuration de la mise en page
    fig.update_layout(
        title=dict(
            text=f"🗺️ Planisphère des Espèces Marines - Grille 50×50<br><sub>🎨 {color_label} | 📊 {len(vis_data)} cellules avec données</sub>",
            x=0.5,
            font=dict(size=18)
        ),
        xaxis=dict(
            title="Longitude",
            tickmode='array',
            tickvals=list(range(0, 50, 5)),
            ticktext=lon_labels,
            side='bottom'
        ),
        yaxis=dict(
            title="Latitude", 
            tickmode='array',
            tickvals=list(range(0, 50, 5)),
            ticktext=lat_labels,
            autorange='reversed'  # Pour que le Nord soit en haut
        ),
        width=1000,
        height=600,
        font=dict(size=12)
    )
    
    print(f"✅ Planisphère créé avec {len(vis_data)} cellules colorées")
    print(f"🎨 Couleurs basées sur: {color_label}")
    print("🖱️ Survolez les cellules pour voir les détails!")
    
    # Afficher le planisphère
    fig.show()
    
    # Statistiques du planisphère
    valid_data = vis_data[color_var].dropna()
    if len(valid_data) > 0:
        print(f"\n📊 STATISTIQUES DU PLANISPHÈRE:")
        print(f"   🗺️ Cellules avec données: {len(vis_data)}")
        print(f"   📊 {color_label}: min={valid_data.min():.2f}, max={valid_data.max():.2f}")
        print(f"   📈 Couverture mondiale: {len(vis_data)/2500*100:.1f}%")
        print(f"   🌍 Chaque cellule = 3.6° lat × 7.2° lon")

🗺️ Création du planisphère avec rectangles colorés...
📊 Création du planisphère avec 1906 cellules
🎨 Variable de couleur: Température Moyenne (°C)
✅ Planisphère créé avec 1906 cellules colorées
🎨 Couleurs basées sur: Température Moyenne (°C)
🖱️ Survolez les cellules pour voir les détails!



📊 STATISTIQUES DU PLANISPHÈRE:
   🗺️ Cellules avec données: 1906
   📊 Température Moyenne (°C): min=-0.55, max=25.69
   📈 Couverture mondiale: 76.2%
   🌍 Chaque cellule = 3.6° lat × 7.2° lon


In [9]:
# Résumé final et export des données
print("📋 Résumé final de l'analyse...")

# Vérifier la disponibilité des données
has_sample = 'df_sample' in locals() and len(df_sample) > 0
has_grid = 'grid_summary' in locals() and len(grid_summary) > 0
has_original = 'df_full' in locals() if 'df_full' in locals() else False

print(f"\n🎯 RÉSUMÉ DE L'ANALYSE:")

if has_original:
    print(f"   📊 Dataset original: Fichier volumineux (16+ millions de lignes)")
else:
    print(f"   📊 Dataset original: Non chargé (fichier trop volumineux)")

if has_sample:
    print(f"   🎲 Échantillon analysé: {len(df_sample):,} observations")
else:
    print(f"   🎲 Échantillon analysé: Échec de l'échantillonnage")

if 'grid_size' in locals():
    print(f"   🌐 Grille créée: {grid_size}×{grid_size} = {grid_size*grid_size:,} cellules")
else:
    print(f"   🌐 Grille créée: 50×50 = 2,500 cellules (valeur par défaut)")

if has_grid:
    grid_coverage = len(grid_summary) / (grid_size * grid_size if 'grid_size' in locals() else 2500) * 100
    print(f"   📍 Cellules avec données: {len(grid_summary):,} ({grid_coverage:.1f}%)")
    
    if 'df_grid' in locals() and len(df_grid) > 0:
        print(f"   🐠 Espèces uniques observées: {df_grid['species'].nunique():,}")
        print(f"   🧬 Genres uniques observés: {df_grid['genus'].nunique():,}")
else:
    print(f"   📍 Cellules avec données: Aucune (échec de l'agrégation)")

# Statistiques par zone climatique si disponibles
if has_grid and 'zone' in grid_summary.columns:
    print(f"\n🌍 RÉPARTITION PAR ZONES CLIMATIQUES:")
    for zone in ['Arctique', 'Tempérée Nord', 'Tropicale', 'Tempérée Sud', 'Antarctique']:
        zone_data = grid_summary[grid_summary['zone'] == zone]
        if len(zone_data) > 0:
            print(f"   {zone:>15}: {len(zone_data):3d} cellules, {zone_data['observation_count'].sum():,} observations")

# Variables environnementales analysées
if 'available_cols' in locals() and len(available_cols) > 0:
    print(f"\n🔬 VARIABLES ENVIRONNEMENTALES ANALYSÉES:")
    for i, col in enumerate(available_cols, 1):
        if has_grid and col in grid_summary.columns:
            valid_count = grid_summary[col].notna().sum()
            print(f"   {i}. {col}: {valid_count:,} cellules avec données")
else:
    print(f"\n🔬 VARIABLES ENVIRONNEMENTALES: Aucune variable numérique disponible")

# Recommandations pour l'utilisation
print(f"\n💡 UTILISATION DE LA GRILLE:")
if has_grid:
    print(f"   🖱️ Survolez les points sur les cartes pour voir les détails")
    lat_step = 180 / (grid_size if 'grid_size' in locals() else 50)
    lon_step = 360 / (grid_size if 'grid_size' in locals() else 50)
    print(f"   🔍 Chaque cellule représente une zone de {lat_step:.2f}° × {lon_step:.2f}°")
    print(f"   📈 Les couleurs indiquent les valeurs moyennes des variables")
    print(f"   📏 La taille des points indique le nombre d'observations")
else:
    print(f"   ❌ Grille non disponible - vérifiez les données source")

# Diagnostics et recommandations
print(f"\n🔧 DIAGNOSTICS:")
if not has_sample:
    print(f"   ❌ Échantillonnage échoué - vérifiez le chemin du fichier CSV")
    print(f"   💡 Assurez-vous que 'Scrapping/marine_species_expanded.csv' existe")
elif not has_grid:
    print(f"   ❌ Agrégation échouée - vérifiez la qualité des coordonnées")
    print(f"   💡 Vérifiez les colonnes 'center_lat' et 'center_long'")
else:
    print(f"   ✅ Analyse réussie avec {len(grid_summary)} cellules de données")

# Option d'export des données de grille
if has_grid:
    print(f"\n💾 Export des données de grille...")
    try:
        # Sauvegarder les données de grille
        export_file = 'marine_species_grid_50x50.csv'
        grid_summary.to_csv(export_file, index=False)
        print(f"✅ Données de grille exportées vers: {export_file}")
        print(f"   📁 Ce fichier contient {len(grid_summary)} cellules avec leurs statistiques")
    except Exception as e:
        print(f"⚠️ Erreur lors de l'export: {e}")
else:
    print(f"\n💾 Export: Impossible (aucune donnée de grille disponible)")

# Message final
if has_grid:
    print(f"\n🎉 Analyse terminée! Vous disposez maintenant d'une grille interactive 50×50")
    print(f"   du monde entier avec les données moyennées de vos espèces marines.")
    
    print(f"\n📚 Pour utiliser cette grille:")
    print(f"   1. Chaque point représente le centre d'une cellule")
    print(f"   2. Survolez pour voir les détails (température, profondeur, etc.)")
    print(f"   3. Les différentes cartes montrent différents aspects des données")
    print(f"   4. La heatmap montre la densité globale d'observations")
else:
    print(f"\n❌ Analyse incomplète. Vérifiez:")
    print(f"   1. L'existence du fichier 'Scrapping/marine_species_expanded.csv'")
    print(f"   2. La présence des colonnes 'center_lat' et 'center_long'")
    print(f"   3. La qualité des données (valeurs non nulles)")
    print(f"   4. L'espace mémoire disponible")

print(f"\n📊 STATUT FINAL: {'✅ SUCCÈS' if has_grid else '❌ ÉCHEC PARTIEL'}")

# Définitions DÉFINITIVES des fonctions de grille (pour éviter les conflits)
def grid_index(coord, min_val, max_val, grid_size):
    """Calculer l'index de grille pour une coordonnée"""
    if pd.isna(coord):
        return None
    
    # Normaliser la coordonnée entre 0 et 1
    normalized = (coord - min_val) / (max_val - min_val)
    
    # Calculer l'index de la grille (0 à grid_size-1)
    index = int(normalized * grid_size)
    index = max(0, min(index, grid_size - 1))
    
    return index

def get_cell_center_coords(grid_index, min_val, max_val, grid_size):
    """Obtenir le centre d'une cellule de grille - VERSION DÉFINITIVE"""
    if pd.isna(grid_index):
        return None
    cell_size = (max_val - min_val) / grid_size
    return min_val + (grid_index + 0.5) * cell_size

def get_cell_center(grid_index, min_val, max_val, grid_size):
    """Obtenir le centre d'une cellule de grille - ALIAS de get_cell_center_coords"""
    return get_cell_center_coords(grid_index, min_val, max_val, grid_size)

# Fonction pour obtenir les coordonnées du centre à partir des indices lat/lon
def get_grid_center_coords(lat_idx, lon_idx, lat_min=-90, lat_max=90, lon_min=-180, lon_max=180, grid_size=50):
    """Obtenir les coordonnées du centre à partir des indices de grille"""
    lat_center = get_cell_center_coords(lat_idx, lat_min, lat_max, grid_size)
    lon_center = get_cell_center_coords(lon_idx, lon_min, lon_max, grid_size)
    return lat_center, lon_center

print("✅ Toutes les fonctions de grille redéfinies correctement")
print("🔧 Fonctions disponibles:")
print("   • grid_index(coord, min_val, max_val, grid_size)")
print("   • get_cell_center_coords(grid_index, min_val, max_val, grid_size)")
print("   • get_cell_center(grid_index, min_val, max_val, grid_size)")
print("   • get_grid_center_coords(lat_idx, lon_idx, ...)")

# Test rapide pour vérifier que ça marche
test_lat = get_cell_center_coords(25, -90, 90, 50)  # Cellule du milieu
test_lon = get_cell_center_coords(25, -180, 180, 50)
print(f"🧪 Test: Cellule centrale (25,25) → lat={test_lat:.1f}°, lon={test_lon:.1f}°")

📋 Résumé final de l'analyse...

🎯 RÉSUMÉ DE L'ANALYSE:
   📊 Dataset original: Non chargé (fichier trop volumineux)
   🎲 Échantillon analysé: 499,281 observations
   🌐 Grille créée: 50×50 = 2,500 cellules
   📍 Cellules avec données: 1,906 (76.2%)
   🐠 Espèces uniques observées: 1,669
   🧬 Genres uniques observés: 1,050

🔬 VARIABLES ENVIRONNEMENTALES ANALYSÉES:
   1. temperature_param_mean: 1,896 cellules avec données
   2. depth_param_mean: 1,896 cellules avec données
   3. salinity_param_mean: 1,896 cellules avec données
   4. primary_production_param_mean: 1,896 cellules avec données
   5. dissolved_oxygen_param_mean: 1,896 cellules avec données
   6. distance_to_land_param_mean: 1,896 cellules avec données
   7. overall_probability: 1,906 cellules avec données

💡 UTILISATION DE LA GRILLE:
   🖱️ Survolez les points sur les cartes pour voir les détails
   🔍 Chaque cellule représente une zone de 3.60° × 7.20°
   📈 Les couleurs indiquent les valeurs moyennes des variables
   📏 La taille 

# 🔍 Analyse Spécifique par Espèce

Cette section permet de créer une carte de distribution pour une espèce spécifique en utilisant les fichiers CSV individuels du dossier `Complete CSVs/`.

## 📋 Instructions
1. Exécutez d'abord la cellule ci-dessous pour voir la liste des espèces disponibles
2. Entrez le nom exact d'une espèce (ex: "Actinia_equina")
3. La carte montrera la distribution de cette espèce avec:
   - **Couleur** : Probabilité moyenne de présence dans chaque cellule
   - **Taille** : Nombre d'observations dans la cellule
   - **Détails au survol** : Coordonnées, probabilités, statistiques

In [10]:
# Liste des espèces disponibles dans le dossier Complete CSVs
import os
import glob

print("🔍 Recherche des espèces disponibles...")

# Chercher les fichiers CSV dans le dossier Complete CSVs
csv_folder = "Complete CSVs"
available_species = []

if os.path.exists(csv_folder):
    # Lister tous les fichiers CSV
    csv_files = glob.glob(os.path.join(csv_folder, "*.csv"))
    
    # Extraire les noms d'espèces (sans le chemin et l'extension)
    available_species = [os.path.splitext(os.path.basename(f))[0] for f in csv_files]
    available_species.sort()  # Trier alphabétiquement
    
    print(f"✅ Trouvé {len(available_species)} espèces disponibles")
    print(f"📁 Dossier: {csv_folder}")
    
    # Afficher les 20 premières espèces comme exemples
    print("\n🐠 Exemples d'espèces disponibles (20 premières):")
    for i, species in enumerate(available_species[:20], 1):
        print(f"   {i:2d}. {species}")
    
    if len(available_species) > 20:
        print(f"   ... et {len(available_species) - 20} autres espèces")
    
    # Quelques suggestions populaires
    popular_species = ["Actinia_equina", "Abudefduf_vaigiensis", "Acanthaster_planci"]
    found_popular = [s for s in popular_species if s in available_species]
    
    if found_popular:
        print(f"\n🌟 Espèces suggérées pour commencer:")
        for species in found_popular:
            print(f"   • {species}")
    
else:
    print(f"❌ Dossier '{csv_folder}' non trouvé")
    print(f"💡 Assurez-vous que le dossier existe avec les fichiers CSV des espèces")
    available_species = []

print(f"\n📝 Pour analyser une espèce, utilisez son nom exact dans la cellule suivante.")

🔍 Recherche des espèces disponibles...
✅ Trouvé 2049 espèces disponibles
📁 Dossier: Complete CSVs

🐠 Exemples d'espèces disponibles (20 premières):
    1. Ablennes_hians
    2. Abraliopsis_morisii
    3. Abudefduf_vaigiensis
    4. Acanthaster_planci
    5. Acanthochaenus_luetkenii
    6. Acanthochromis_polyacanthus
    7. Acanthocybium_solandri
    8. Acanthogobius_flavimanus
    9. Acanthopagrus_arabicus
   10. Acanthopagrus_australis
   11. Acanthopagrus_bifasciatus
   12. Acanthopagrus_butcheri
   13. Acanthopagrus_latus
   14. Acanthopagrus_schlegelii
   15. Acanthurus_bahianus
   16. Acanthurus_blochii
   17. Acanthurus_dussumieri
   18. Acanthurus_leucosternon
   19. Acanthurus_lineatus
   20. Acanthurus_nigricans
   ... et 2029 autres espèces

🌟 Espèces suggérées pour commencer:
   • Actinia_equina
   • Abudefduf_vaigiensis
   • Acanthaster_planci

📝 Pour analyser une espèce, utilisez son nom exact dans la cellule suivante.


In [11]:
# Analyse d'une espèce spécifique
# 🔧 CONFIGURATION - Modifiez cette ligne avec le nom de l'espèce souhaitée
# species_name = "Acanthopagrus_schlegelii"  # ⬅️ CHANGEZ ICI pour une autre espèce
species_name = "Katsuwonus_pelamis"  # ⬅️ CHANGEZ ICI pour une autre espèce

# S'assurer que tous les modules nécessaires sont importés
try:
    import pandas as pd
    import numpy as np
    import os
    import glob
except ImportError as e:
    print(f"❌ Erreur d'import: {e}")
    print("💡 Assurez-vous que pandas, numpy sont installés")

# S'assurer que les fonctions de grille sont définies
if 'assign_to_grid' not in locals():
    # Redéfinir les fonctions de grille si nécessaires
    def assign_to_grid(coord, min_val, max_val, grid_size):
        """Assigner une coordonnée à une cellule de grille"""
        if pd.isna(coord):
            return None, None
        
        # Normaliser la coordonnée
        normalized = (coord - min_val) / (max_val - min_val)
        
        # Calculer l'index de la grille (0 à grid_size-1)
        grid_index = int(normalized * grid_size)
        grid_index = max(0, min(grid_index, grid_size - 1))
        
        return grid_index, coord

    def get_cell_center(grid_index, min_val, max_val, grid_size):
        """Obtenir le centre d'une cellule de grille"""
        cell_size = (max_val - min_val) / grid_size
        return min_val + (grid_index + 0.5) * cell_size

    def get_cell_bounds(lat_idx, lon_idx):
        """Calculer les limites d'une cellule de grille"""
        lat_step = 180 / grid_size  # 3.6° par cellule
        lon_step = 360 / grid_size  # 7.2° par cellule
        
        lat_south = -90 + lat_idx * lat_step
        lat_north = -90 + (lat_idx + 1) * lat_step
        lon_west = -180 + lon_idx * lon_step
        lon_east = -180 + (lon_idx + 1) * lon_step
        return lat_south, lat_north, lon_west, lon_east

# Définir grid_size si pas déjà défini
if 'grid_size' not in locals():
    grid_size = 50

# Paramètres d'affichage
show_details = True  # Afficher les détails de traitement
min_observations = 1  # Nombre minimum d'observations par cellule pour l'affichage

print(f"🎯 Analyse de l'espèce: {species_name}")
print(f"📁 Recherche du fichier: Complete CSVs/{species_name}.csv")

# Vérifier la disponibilité des espèces si la liste n'est pas déjà chargée
if 'available_species' not in locals():
    print("📂 Chargement de la liste des espèces disponibles...")
    csv_folder = "Complete CSVs"
    available_species = []
    
    if os.path.exists(csv_folder):
        csv_files = glob.glob(os.path.join(csv_folder, "*.csv"))
        available_species = [os.path.splitext(os.path.basename(f))[0] for f in csv_files]

# Vérifier si l'espèce est disponible
if available_species and species_name not in available_species:
    print(f"❌ Espèce '{species_name}' non trouvée")
    print(f"💡 Espèces disponibles: {len(available_species)} au total")
    print(f"🔍 Vérifiez l'orthographe ou choisissez parmi les espèces listées ci-dessus")
else:
    try:
        # Charger les données de l'espèce
        species_file = os.path.join("Complete CSVs", f"{species_name}.csv")
        
        if show_details:
            print(f"📊 Chargement des données...")
        
        # Lire le fichier CSV de l'espèce avec gestion du format spécial
        # Les fichiers d'espèces ont des métadonnées au début, les vraies données commencent vers la ligne 31
        try:
            # D'abord, trouver où commencent les vraies données (ligne avec "Genus,Species,...")
            with open(species_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            header_line = None
            for i, line in enumerate(lines):
                if line.startswith('Genus,Species,'):
                    header_line = i
                    break
            
            if header_line is None:
                # Fallback: essayer de lire normalement avec gestion d'erreurs
                df_species = pd.read_csv(species_file, on_bad_lines='skip')
            else:
                # Lire à partir de la ligne d'en-tête trouvée avec gestion d'erreurs
                df_species = pd.read_csv(species_file, skiprows=header_line, on_bad_lines='skip')
                
        except UnicodeDecodeError:
            # Essayer avec un autre encodage
            try:
                with open(species_file, 'r', encoding='latin-1') as f:
                    lines = f.readlines()
                
                header_line = None
                for i, line in enumerate(lines):
                    if line.startswith('Genus,Species,'):
                        header_line = i
                        break
                
                if header_line is None:
                    df_species = pd.read_csv(species_file, encoding='latin-1', on_bad_lines='skip')
                else:
                    df_species = pd.read_csv(species_file, skiprows=header_line, encoding='latin-1', on_bad_lines='skip')
            except:
                # Dernier recours avec paramètres très permissifs
                df_species = pd.read_csv(species_file, sep=',', on_bad_lines='skip', quoting=1, error_bad_lines=False)
        
        print(f"✅ Données chargées: {len(df_species):,} observations")
        
        # Afficher les premières lignes pour comprendre la structure
        if show_details:
            print(f"\n📋 Structure des données:")
            print(f"   Colonnes: {list(df_species.columns)}")
            print(f"   Premières lignes:")
            print(df_species.head(3))
        
        # Vérifier les colonnes nécessaires et adapter aux noms utilisés dans les fichiers d'espèces
        # Les fichiers d'espèces utilisent "Center Lat" et "Center Long"
        if 'Center Lat' in df_species.columns and 'Center Long' in df_species.columns:
            # Renommer pour correspondre au code existant  
            df_species = df_species.rename(columns={
                'Center Lat': 'center_lat',
                'Center Long': 'center_long'
            })
            required_cols = ['center_lat', 'center_long']
            missing_cols = []
        else:
            required_cols = ['center_lat', 'center_long']
            missing_cols = [col for col in required_cols if col not in df_species.columns]
        
        if missing_cols:
            print(f"❌ Colonnes manquantes: {missing_cols}")
            print(f"💡 Colonnes disponibles: {list(df_species.columns)}")
        else:
            # Nettoyer les données
            df_species_clean = df_species.dropna(subset=['center_lat', 'center_long'])
            
            print(f"🧹 Données nettoyées: {len(df_species_clean):,} observations avec coordonnées valides")
            
            if len(df_species_clean) == 0:
                print(f"❌ Aucune observation avec coordonnées valides")
            else:
                # Assigner les observations aux cellules de la grille
                if show_details:
                    print(f"🗺️ Attribution aux cellules de la grille...")
                
                df_species_clean['grid_lat'] = df_species_clean['center_lat'].apply(
                    lambda lat: grid_index(lat, -90, 90, grid_size)
                )
                df_species_clean['grid_lon'] = df_species_clean['center_long'].apply(
                    lambda lon: grid_index(lon, -180, 180, grid_size)
                )
                
                # Créer l'identifiant de cellule
                df_species_clean['cell_id'] = df_species_clean['grid_lat'].astype(str) + '_' + df_species_clean['grid_lon'].astype(str)
                
                # Compter les observations par cellule et calculer les probabilités moyennes
                agg_dict = {
                    'center_lat': ['mean', 'count'],
                    'center_long': 'mean'
                }
                
                # Ajouter la probabilité globale si elle existe
                if 'Overall Probability' in df_species_clean.columns:
                    agg_dict['Overall Probability'] = ['mean', 'std']
                
                species_grid = df_species_clean.groupby(['grid_lat', 'grid_lon', 'cell_id']).agg(agg_dict).reset_index()
                
                # Simplifier les noms de colonnes
                if 'Overall Probability' in df_species_clean.columns:
                    species_grid.columns = ['grid_lat', 'grid_lon', 'cell_id', 'mean_lat', 'observation_count', 'mean_lon', 'mean_probability', 'std_probability']
                else:
                    species_grid.columns = ['grid_lat', 'grid_lon', 'cell_id', 'mean_lat', 'observation_count', 'mean_lon']
                
                # Calculer les centres des cellules
                species_grid['cell_center_lat'] = species_grid['grid_lat'].apply(
                    lambda i: get_cell_center(i, -90, 90, grid_size)
                )
                species_grid['cell_center_lon'] = species_grid['grid_lon'].apply(
                    lambda i: get_cell_center(i, -180, 180, grid_size)
                )
                
                # Calculer la probabilité de présence
                if 'mean_probability' in species_grid.columns:
                    # Utiliser les vraies probabilités moyennes d'AquaMaps
                    species_grid['presence_probability'] = species_grid['mean_probability']
                else:
                    # Fallback: utiliser le nombre d'observations relatif
                    max_obs = species_grid['observation_count'].max()
                    species_grid['presence_probability'] = species_grid['observation_count'] / max_obs
                
                # Filtrer par nombre minimum d'observations
                species_grid_filtered = species_grid[species_grid['observation_count'] >= min_observations]
                
                print(f"📊 Résultats de l'agrégation:")
                print(f"   🔢 Cellules avec données: {len(species_grid)}")
                print(f"   📍 Cellules affichées (≥{min_observations} obs): {len(species_grid_filtered)}")
                print(f"   📈 Observations par cellule: min={species_grid['observation_count'].min()}, max={species_grid['observation_count'].max()}")
                
                # Afficher les statistiques de probabilité si disponibles
                if 'mean_probability' in species_grid.columns:
                    print(f"   🎯 Probabilité AquaMaps: min={species_grid['mean_probability'].min():.3f}, max={species_grid['mean_probability'].max():.3f}")
                    print(f"   📊 Probabilité moyenne: {species_grid['mean_probability'].mean():.3f} ± {species_grid['mean_probability'].std():.3f}")
                
                print(f"   🌍 Couverture géographique:")
                print(f"      Latitude: {species_grid['mean_lat'].min():.2f}° à {species_grid['mean_lat'].max():.2f}°")
                print(f"      Longitude: {species_grid['mean_lon'].min():.2f}° à {species_grid['mean_lon'].max():.2f}°")
                
    except FileNotFoundError:
        print(f"❌ Fichier non trouvé: {species_file}")
        print(f"💡 Vérifiez que le fichier existe dans le dossier 'Complete CSVs'")
    except Exception as e:
        print(f"❌ Erreur lors du chargement: {e}")
        print(f"💡 Vérifiez le format du fichier CSV")

🎯 Analyse de l'espèce: Katsuwonus_pelamis
📁 Recherche du fichier: Complete CSVs/Katsuwonus_pelamis.csv
📊 Chargement des données...
✅ Données chargées: 110,170 observations

📋 Structure des données:
   Colonnes: ['Genus', 'Species', 'Center Lat', 'Center Long', 'C-Square Code', 'Overall Probability']
   Premières lignes:
        Genus  Species  Center Lat  Center Long C-Square Code  \
0  Katsuwonus  pelamis       61.25      -150.25    7615:110:1   
1  Katsuwonus  pelamis       60.75      -152.25    7615:102:3   
2  Katsuwonus  pelamis       60.25      -152.75    7615:102:2   

   Overall Probability  
0                 0.01  
1                 0.03  
2                 0.06  
🧹 Données nettoyées: 110,169 observations avec coordonnées valides
🗺️ Attribution aux cellules de la grille...
📊 Résultats de l'agrégation:
   🔢 Cellules avec données: 1213
   📍 Cellules affichées (≥1 obs): 1213
   📈 Observations par cellule: min=1, max=120
   🎯 Probabilité AquaMaps: min=0.010, max=1.000
   📊 Probab

In [12]:
# Créer la carte interactive pour l'espèce spécifique

# S'assurer que plotly est importé
try:
    import plotly.graph_objects as go
    import numpy as np
except ImportError as e:
    print(f"❌ Erreur d'import Plotly: {e}")
    print("💡 Installez plotly avec: pip install plotly")

if 'species_grid_filtered' in locals() and len(species_grid_filtered) > 0:
    print(f"🗺️ Création de la carte de distribution pour {species_name}...")
    
    # Créer la carte interactive avec Plotly
    fig_species = go.Figure()
    
    # Déterminer le titre de la colorbar selon le type de probabilité
    colorbar_title = "Probabilité AquaMaps" if 'mean_probability' in species_grid_filtered.columns else "Probabilité Relative"
    
    # Ajouter les points de distribution
    fig_species.add_trace(go.Scattermapbox(
        lat=species_grid_filtered['cell_center_lat'],
        lon=species_grid_filtered['cell_center_lon'],
        mode='markers',
        marker=dict(
            size=np.sqrt(species_grid_filtered['observation_count']) * 3 + 5,  # Taille basée sur le nombre d'observations
            color=species_grid_filtered['presence_probability'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(
                title=dict(
                    text=colorbar_title,
                    font=dict(size=14)
                ),
                thickness=15,
                len=0.7
            ),
            cmin=0,
            cmax=1,
            opacity=0.8
        ),
        text=[
            f"<b>{species_name.replace('_', ' ')}</b><br>" +
            f"🔢 Cellule: ({row['grid_lat']}, {row['grid_lon']})<br>" +
            f"📍 Centre: ({row['cell_center_lat']:.2f}°, {row['cell_center_lon']:.2f}°)<br>" +
            f"🔬 Observations: {row['observation_count']:,}<br>" +
            (f"🎯 Probabilité AquaMaps: {row['mean_probability']:.3f}<br>" if 'mean_probability' in row and pd.notna(row['mean_probability']) else "") +
            (f"📊 Écart-type probabilité: {row['std_probability']:.3f}<br>" if 'std_probability' in row and pd.notna(row['std_probability']) else "") +
            f"📈 Probabilité relative: {row['presence_probability']:.2%}<br>" +
            f"🌐 Lat moyenne: {row['mean_lat']:.3f}°<br>" +
            f"🌐 Lon moyenne: {row['mean_lon']:.3f}°"
            for _, row in species_grid_filtered.iterrows()
        ],
        hovertemplate='%{text}<extra></extra>',
        name=f'Distribution de {species_name}'
    ))
    
    # Configuration de la carte
    fig_species.update_layout(
        title=dict(
            text=f"🐠 Distribution de {species_name.replace('_', ' ')}<br>" +
                 f"<sub>Grille {grid_size}×{grid_size} - {len(species_grid_filtered)} cellules avec données</sub>",
            x=0.5,
            font=dict(size=16)
        ),
        mapbox=dict(
            style="open-street-map",
            center=dict(
                lat=species_grid_filtered['cell_center_lat'].mean(),
                lon=species_grid_filtered['cell_center_lon'].mean()
            ),
            zoom=2
        ),
        width=1000,
        height=600,
        margin=dict(l=0, r=0, t=80, b=0)
    )
    
    # Afficher la carte
    fig_species.show()
    
    # Statistiques de la distribution
    print(f"\n📊 STATISTIQUES DE DISTRIBUTION - {species_name}:")
    print(f"   🔢 Total des observations: {species_grid_filtered['observation_count'].sum():,}")
    print(f"   📍 Cellules occupées: {len(species_grid_filtered)} sur {grid_size*grid_size} ({len(species_grid_filtered)/(grid_size*grid_size)*100:.2f}%)")
    print(f"   📈 Observations par cellule: {species_grid_filtered['observation_count'].mean():.1f} ± {species_grid_filtered['observation_count'].std():.1f}")
    print(f"   🎯 Cellule la plus dense: {species_grid_filtered['observation_count'].max():,} observations")
    print(f"   🌍 Répartition géographique:")
    print(f"      📏 Latitude: {species_grid_filtered['mean_lat'].min():.2f}° à {species_grid_filtered['mean_lat'].max():.2f}°")
    print(f"      📏 Longitude: {species_grid_filtered['mean_lon'].min():.2f}° à {species_grid_filtered['mean_lon'].max():.2f}°")
    
    # Analyse des zones de forte concentration
    top_cells = species_grid_filtered.nlargest(5, 'observation_count')
    print(f"\n🏆 TOP 5 DES ZONES DE FORTE CONCENTRATION:")
    for i, (_, row) in enumerate(top_cells.iterrows(), 1):
        print(f"   {i:2d}. Cellule ({row['grid_lat']}, {row['grid_lon']}): {row['observation_count']:,} observations")
        print(f"      📍 Position: {row['cell_center_lat']:.2f}°N, {row['cell_center_lon']:.2f}°E")
        print(f"      📊 Probabilité: {row['presence_probability']:.1%}")
    
    print(f"\n💡 INTERPRÉTATION DE LA CARTE:")
    print(f"   🎨 Couleur: Plus c'est vert/jaune, plus la probabilité de présence est élevée")
    print(f"   📏 Taille: Plus le point est gros, plus il y a d'observations")
    print(f"   🖱️ Survolez les points pour voir les détails de chaque cellule")
    
else:
    print(f"❌ Impossible de créer la carte pour {species_name}")
    if 'df_species_clean' in locals():
        print(f"💡 Données disponibles mais aucune cellule ne répond aux critères (min {min_observations} observations)")
    else:
        print(f"💡 Aucune donnée valide trouvée pour cette espèce")

🗺️ Création de la carte de distribution pour Katsuwonus_pelamis...



📊 STATISTIQUES DE DISTRIBUTION - Katsuwonus_pelamis:
   🔢 Total des observations: 110,169
   📍 Cellules occupées: 1213 sur 2500 (48.52%)
   📈 Observations par cellule: 90.8 ± 28.4
   🎯 Cellule la plus dense: 120 observations
   🌍 Répartition géographique:
      📏 Latitude: -45.62° à 62.58°
      📏 Longitude: -177.27° à 176.63°

🏆 TOP 5 DES ZONES DE FORTE CONCENTRATION:
    1. Cellule (12, 1): 120 observations
      📍 Position: -45.00°N, -169.20°E
      📊 Probabilité: 54.9%
    2. Cellule (12, 3): 120 observations
      📍 Position: -45.00°N, -154.80°E
      📊 Probabilité: 48.5%
    3. Cellule (12, 6): 120 observations
      📍 Position: -45.00°N, -133.20°E
      📊 Probabilité: 42.1%
    4. Cellule (12, 8): 120 observations
      📍 Position: -45.00°N, -118.80°E
      📊 Probabilité: 40.4%
    5. Cellule (12, 11): 120 observations
      📍 Position: -45.00°N, -97.20°E
      📊 Probabilité: 39.2%

💡 INTERPRÉTATION DE LA CARTE:
   🎨 Couleur: Plus c'est vert/jaune, plus la probabilité de présen

In [13]:
# Créer une heatmap de la distribution de l'espèce
if 'species_grid' in locals() and len(species_grid) > 0:
    print(f"🔥 Création de la heatmap de distribution pour {species_name}...")
    
    # Créer une matrice pour la heatmap
    heatmap_matrix = np.zeros((grid_size, grid_size))
    
    # Remplir la matrice avec les données d'observation
    for _, row in species_grid.iterrows():
        i, j = int(row['grid_lat']), int(row['grid_lon'])
        if 0 <= i < grid_size and 0 <= j < grid_size:
            heatmap_matrix[i, j] = row['observation_count']
    
    # Créer les étiquettes des axes en utilisant notre fonction get_cell_center_coords
    lat_labels = [f"{get_cell_center_coords(i, -90, 90, grid_size):.1f}°" for i in range(grid_size)]
    lon_labels = [f"{get_cell_center_coords(j, -180, 180, grid_size):.1f}°" for j in range(grid_size)]
    
    # Créer la heatmap avec Plotly
    fig_heatmap = go.Figure(data=go.Heatmap(
        z=heatmap_matrix,
        x=lon_labels,
        y=lat_labels,
        colorscale='Viridis',
        colorbar=dict(
            title=dict(
                text="Nombre d'Observations",
                font=dict(size=12)
            ),
            thickness=15,
            len=0.8
        ),
        hoverongaps=False,
        hovertemplate=(
            f"<b>{species_name}</b><br>" +
            "Latitude: %{y}<br>" +
            "Longitude: %{x}<br>" +
            "Observations: %{z}<br>" +
            "<extra></extra>"
        )
    ))
    
    # Configuration de la heatmap
    fig_heatmap.update_layout(
        title=dict(
            text=f"🔥 Heatmap de Distribution - {species_name.replace('_', ' ')}<br>" +
                 f"<sub>Grille {grid_size}×{grid_size} - Densité d'observations par cellule</sub>",
            x=0.5,
            font=dict(size=16)
        ),
        xaxis=dict(
            title="Longitude",
            tickangle=45,
            dtick=5  # Afficher une étiquette tous les 5 indices
        ),
        yaxis=dict(
            title="Latitude",
            dtick=5
        ),
        width=900,
        height=600,
        margin=dict(l=60, r=60, t=100, b=80)
    )
    
    # Afficher la heatmap
    fig_heatmap.show()
    
    # Statistiques de la heatmap
    non_zero_cells = np.count_nonzero(heatmap_matrix)
    total_observations = np.sum(heatmap_matrix)
    max_density = np.max(heatmap_matrix)
    
    print(f"\n🔥 STATISTIQUES DE LA HEATMAP - {species_name}:")
    print(f"   📊 Cellules avec observations: {non_zero_cells} sur {grid_size*grid_size} ({non_zero_cells/(grid_size*grid_size)*100:.2f}%)")
    print(f"   🔢 Total des observations: {int(total_observations):,}")
    print(f"   📈 Densité maximale: {int(max_density):,} observations par cellule")
    print(f"   📏 Densité moyenne (cellules actives): {total_observations/non_zero_cells if non_zero_cells > 0 else 0:.1f} observations/cellule")
    
    # Trouver les zones de plus forte densité
    max_positions = np.where(heatmap_matrix == max_density)
    print(f"\n🎯 ZONES DE DENSITÉ MAXIMALE ({int(max_density)} observations):")
    for i in range(len(max_positions[0])):
        lat_idx, lon_idx = max_positions[0][i], max_positions[1][i]
    center_lat = get_cell_center_coords(lat_idx, -90, 90, grid_size)
    center_lon = get_cell_center_coords(lon_idx, -180, 180, grid_size)
    print(f"   📍 Cellule ({lat_idx}, {lon_idx}): {center_lat:.2f}°N, {center_lon:.2f}°E")
    
    print(f"\n💡 UTILISATION DE LA HEATMAP:")
    print(f"   🎨 Plus la couleur est claire/jaune, plus il y a d'observations")
    print(f"   🖱️ Survolez les cellules pour voir le nombre exact d'observations")
    print(f"   📊 Cette vue permet d'identifier rapidement les hotspots de l'espèce")
    
else:
    print(f"❌ Impossible de créer la heatmap pour {species_name}")
    print(f"💡 Assurez-vous que les données de grille ont été générées correctement")

🔥 Création de la heatmap de distribution pour Katsuwonus_pelamis...



🔥 STATISTIQUES DE LA HEATMAP - Katsuwonus_pelamis:
   📊 Cellules avec observations: 1213 sur 2500 (48.52%)
   🔢 Total des observations: 110,169
   📈 Densité maximale: 120 observations par cellule
   📏 Densité moyenne (cellules actives): 90.8 observations/cellule

🎯 ZONES DE DENSITÉ MAXIMALE (120 observations):
   📍 Cellule (37, 48): 45.00°N, 169.20°E

💡 UTILISATION DE LA HEATMAP:
   🎨 Plus la couleur est claire/jaune, plus il y a d'observations
   🖱️ Survolez les cellules pour voir le nombre exact d'observations
   📊 Cette vue permet d'identifier rapidement les hotspots de l'espèce


# 🤖 Machine Learning : Prédiction de Probabilités de Présence

Cette section implémente un modèle de machine learning pour prédire la probabilité de présence d'espèces marines dans chaque cellule de la grille 50×50.

## 🎯 Objectifs
1. **Prédiction spatiale complète** : Estimer la probabilité dans toutes les 2,500 cellules
2. **Découverte de patterns** : Identifier les relations entre facteurs environnementaux
3. **Validation croisée** : Comparer avec les données AquaMaps existantes
4. **Applications pratiques** : Conservation, pêche, changement climatique

## 📊 Classification en 5 classes
- **Classe 0** : Très improbable (0.0-0.2)
- **Classe 1** : Improbable (0.2-0.4)
- **Classe 2** : Possible (0.4-0.6)
- **Classe 3** : Probable (0.6-0.8)
- **Classe 4** : Très probable (0.8-1.0)

In [14]:
# Configuration et imports pour le Machine Learning
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

print("🤖 Configuration du Machine Learning...")
print("📚 Modèles disponibles : Random Forest, Gradient Boosting")
print("🎯 Objectif : Prédire 5 classes de probabilité de présence")

# Vérifier si nous avons des données d'espèce
if 'species_grid' not in locals() or len(species_grid) == 0:
    print("⚠️ Aucune donnée d'espèce chargée.")
    print("💡 Veuillez d'abord exécuter l'analyse d'une espèce spécifique")
    ml_ready = False
else:
    print(f"✅ Données disponibles : {len(species_grid)} cellules pour {species_name}")
    ml_ready = True

# Fonction pour convertir probabilité en classe
def probability_to_class(prob):
    """Convertit une probabilité (0-1) en classe (0-4)"""
    if prob < 0.2:
        return 0  # Très improbable
    elif prob < 0.4:
        return 1  # Improbable
    elif prob < 0.6:
        return 2  # Possible
    elif prob < 0.8:
        return 3  # Probable
    else:
        return 4  # Très probable

# Fonction inverse pour prédiction
def class_to_probability_range(class_pred):
    """Convertit une classe en gamme de probabilité"""
    ranges = {
        0: "0.0-0.2 (Très improbable)",
        1: "0.2-0.4 (Improbable)", 
        2: "0.4-0.6 (Possible)",
        3: "0.6-0.8 (Probable)",
        4: "0.8-1.0 (Très probable)"
    }
    return ranges.get(class_pred, "Inconnu")

print("\n📋 Classes de probabilité définies:")
for i in range(5):
    print(f"   Classe {i}: {class_to_probability_range(i)}")

🤖 Configuration du Machine Learning...
📚 Modèles disponibles : Random Forest, Gradient Boosting
🎯 Objectif : Prédire 5 classes de probabilité de présence
✅ Données disponibles : 1213 cellules pour Katsuwonus_pelamis

📋 Classes de probabilité définies:
   Classe 0: 0.0-0.2 (Très improbable)
   Classe 1: 0.2-0.4 (Improbable)
   Classe 2: 0.4-0.6 (Possible)
   Classe 3: 0.6-0.8 (Probable)
   Classe 4: 0.8-1.0 (Très probable)


In [15]:
# Préparation des features pour le machine learning
if ml_ready:
    print("🔧 Préparation des features pour le machine learning...")
    
    # Définir les fonctions de grille nécessaires
    def get_cell_center(grid_index, min_val, max_val, grid_size):
        """Obtenir le centre d'une cellule de grille"""
        cell_size = (max_val - min_val) / grid_size
        return min_val + (grid_index + 0.5) * cell_size
    
    def get_cell_bounds(lat_idx, lon_idx):
        """Calculer les limites d'une cellule de grille"""
        lat_step = 180 / grid_size  # 3.6° par cellule
        lon_step = 360 / grid_size  # 7.2° par cellule
        
        lat_south = -90 + lat_idx * lat_step
        lat_north = -90 + (lat_idx + 1) * lat_step
        lon_west = -180 + lon_idx * lon_step
        lon_east = -180 + (lon_idx + 1) * lon_step
        return lat_south, lat_north, lon_west, lon_east
    
    # Créer une grille complète 50x50 pour avoir toutes les cellules
    print("🗺️ Génération de la grille complète 50×50...")
    
    # Générer toutes les combinaisons de cellules possibles
    all_cells = []
    for lat_idx in range(grid_size):
        for lon_idx in range(grid_size):
            # Calculer les coordonnées du centre de la cellule
            center_lat = get_cell_center_coords(lat_idx, -90, 90, grid_size)
            center_lon = get_cell_center_coords(lon_idx, -180, 180, grid_size)
            
            # Calculer les 4 coins de la cellule
            lat_south, lat_north, lon_west, lon_east = get_cell_bounds(lat_idx, lon_idx)
            
            cell_data = {
                'grid_lat': lat_idx,
                'grid_lon': lon_idx,
                'center_lat': center_lat,
                'center_lon': center_lon,
                'lat_south': lat_south,
                'lat_north': lat_north, 
                'lon_west': lon_west,
                'lon_east': lon_east
            }
            all_cells.append(cell_data)
    
    # Convertir en DataFrame
    df_all_cells = pd.DataFrame(all_cells)
    print(f"✅ Grille complète créée : {len(df_all_cells)} cellules")
    
    # Ajouter les données d'observation quand disponibles
    df_ml = df_all_cells.copy()
    
    # Merger avec les données d'espèce existantes
    species_data = species_grid[['grid_lat', 'grid_lon', 'observation_count', 'presence_probability']].copy()
    if 'mean_probability' in species_grid.columns:
        species_data['aquamaps_probability'] = species_grid['mean_probability']
    
    df_ml = df_ml.merge(species_data, on=['grid_lat', 'grid_lon'], how='left')
    
    # Remplir les valeurs manquantes
    df_ml['observation_count'] = df_ml['observation_count'].fillna(0)
    df_ml['presence_probability'] = df_ml['presence_probability'].fillna(0.0)
    if 'aquamaps_probability' in df_ml.columns:
        df_ml['aquamaps_probability'] = df_ml['aquamaps_probability'].fillna(0.0)
    
    print(f"📊 Données fusionnées : {len(df_ml)} cellules total")
    print(f"   • Cellules avec observations : {(df_ml['observation_count'] > 0).sum()}")
    print(f"   • Cellules sans observations : {(df_ml['observation_count'] == 0).sum()}")
    
else:
    print("❌ Préparation des features impossible - données manquantes")

🔧 Préparation des features pour le machine learning...
🗺️ Génération de la grille complète 50×50...
✅ Grille complète créée : 2500 cellules
📊 Données fusionnées : 2500 cellules total
   • Cellules avec observations : 1213
   • Cellules sans observations : 1287


In [16]:
# Création des features environnementales et géographiques
if ml_ready:
    print("🌍 Création des features environnementales et géographiques...")
    
    # S'assurer que les fonctions de grille sont disponibles ici aussi
    if 'get_cell_center' not in locals():
        def get_cell_center(grid_index, min_val, max_val, grid_size):
            """Obtenir le centre d'une cellule de grille"""
            cell_size = (max_val - min_val) / grid_size
            return min_val + (grid_index + 0.5) * cell_size
        
        def get_cell_bounds(lat_idx, lon_idx):
            """Calculer les limites d'une cellule de grille"""
            lat_step = 180 / grid_size  # 3.6° par cellule
            lon_step = 360 / grid_size  # 7.2° par cellule
            
            lat_south = -90 + lat_idx * lat_step
            lat_north = -90 + (lat_idx + 1) * lat_step
            lon_west = -180 + lon_idx * lon_step
            lon_east = -180 + (lon_idx + 1) * lon_step
            return lat_south, lat_north, lon_west, lon_east
    
    # Features géographiques de base
    df_ml['lat_abs'] = np.abs(df_ml['center_lat'])  # Distance à l'équateur
    df_ml['lon_abs'] = np.abs(df_ml['center_lon'])  # Distance au méridien de Greenwich
    
    # Zone climatique basée sur la latitude
    def get_climate_zone(lat):
        lat_abs = abs(lat)
        if lat_abs >= 66.5:
            return 0  # Polaire
        elif lat_abs >= 35:
            return 1  # Tempérée
        else:
            return 2  # Tropicale
    
    df_ml['climate_zone'] = df_ml['center_lat'].apply(get_climate_zone)
    
    # Distance à la côte approximative (simplifiée)
    def estimate_distance_to_coast(lat, lon):
        """Estimation grossière de la distance à la côte"""
        # Zones océaniques principales (très simplifié)
        if abs(lat) > 60:  # Zones polaires
            return 1  # Proche des côtes
        elif -20 <= lat <= 20 and -40 <= lon <= 20:  # Atlantique tropical
            return 3  # Océan ouvert
        elif -60 <= lat <= 60 and 40 <= lon <= 180:  # Indo-Pacifique
            return 2  # Semi-océanique
        else:
            return 1  # Proche des côtes
    
    df_ml['distance_coast_est'] = df_ml.apply(lambda row: estimate_distance_to_coast(row['center_lat'], row['center_lon']), axis=1)
    
    # Profondeur estimée basée sur la position (très simplifié)
    def estimate_depth(lat, lon):
        """Estimation de profondeur basée sur la géographie"""
        lat_abs = abs(lat)
        lon_abs = abs(lon)
        
        # Zones côtières (profondeur faible)
        if lat_abs > 60 or (lat_abs < 30 and lon_abs < 30):
            return np.random.uniform(0, 200)  # Plateau continental
        # Zones océaniques (profondeur élevée)
        else:
            return np.random.uniform(200, 4000)  # Océan profond
    
    np.random.seed(42)  # Pour reproductibilité
    df_ml['depth_estimated'] = df_ml.apply(lambda row: estimate_depth(row['center_lat'], row['center_lon']), axis=1)
    
    # Température estimée basée sur la latitude
    def estimate_temperature(lat):
        """Estimation de température basée sur la latitude"""
        lat_abs = abs(lat)
        if lat_abs >= 66.5:  # Polaire
            return np.random.uniform(-2, 5)
        elif lat_abs >= 35:  # Tempérée
            return np.random.uniform(5, 20)
        else:  # Tropicale
            return np.random.uniform(20, 30)
    
    df_ml['temperature_estimated'] = df_ml['center_lat'].apply(estimate_temperature)
    
    # Salinité estimée (océan ouvert vs côtes)
    def estimate_salinity(distance_coast):
        """Estimation de salinité basée sur la distance à la côte"""
        if distance_coast == 1:  # Côte
            return np.random.uniform(30, 35)
        elif distance_coast == 2:  # Semi-océanique
            return np.random.uniform(34, 36)
        else:  # Océan ouvert
            return np.random.uniform(35, 37)
    
    df_ml['salinity_estimated'] = df_ml['distance_coast_est'].apply(estimate_salinity)
    
    # Features des 4 coins de la cellule
    df_ml['lat_range'] = df_ml['lat_north'] - df_ml['lat_south']
    df_ml['lon_range'] = df_ml['lon_east'] - df_ml['lon_west']
    df_ml['cell_area'] = df_ml['lat_range'] * df_ml['lon_range']  # Aire approximative
    
    print("✅ Features créées :")
    print(f"   • Géographiques : centre, coins, zones climatiques")
    print(f"   • Environnementales : température, profondeur, salinité estimées")
    print(f"   • Spatiales : distance côte, aire de cellule")
    
    # Sélectionner les features pour le modèle
    feature_columns = [
        'center_lat', 'center_lon', 'lat_abs', 'lon_abs',
        'lat_south', 'lat_north', 'lon_west', 'lon_east',
        'climate_zone', 'distance_coast_est',
        'depth_estimated', 'temperature_estimated', 'salinity_estimated',
        'lat_range', 'lon_range', 'cell_area'
    ]
    
    print(f"\n📋 Features sélectionnées ({len(feature_columns)}) :")
    for i, feature in enumerate(feature_columns, 1):
        print(f"   {i:2d}. {feature}")
        
else:
    print("❌ Création des features impossible - données manquantes")

🌍 Création des features environnementales et géographiques...
✅ Features créées :
   • Géographiques : centre, coins, zones climatiques
   • Environnementales : température, profondeur, salinité estimées
   • Spatiales : distance côte, aire de cellule

📋 Features sélectionnées (16) :
    1. center_lat
    2. center_lon
    3. lat_abs
    4. lon_abs
    5. lat_south
    6. lat_north
    7. lon_west
    8. lon_east
    9. climate_zone
   10. distance_coast_est
   11. depth_estimated
   12. temperature_estimated
   13. salinity_estimated
   14. lat_range
   15. lon_range
   16. cell_area


In [17]:
# Entraînement du modèle de machine learning
if ml_ready and 'feature_columns' in locals():
    print("🤖 Entraînement du modèle de machine learning...")
    
    # Préparer les données d'entraînement
    # Utiliser seulement les cellules avec des observations pour l'entraînement
    train_mask = df_ml['observation_count'] > 0
    df_train = df_ml[train_mask].copy()
    
    print(f"📊 Données d'entraînement : {len(df_train)} cellules avec observations")
    
    if len(df_train) < 10:
        print("⚠️ Pas assez de données pour entraîner un modèle robuste")
        print("💡 Essayez avec une espèce ayant plus d'observations")
        ml_trained = False
    else:
        # Préparer X (features) et y (target)
        X = df_train[feature_columns].copy()
        
        # Utiliser les probabilités AquaMaps si disponibles, sinon les probabilités relatives
        if 'aquamaps_probability' in df_train.columns and df_train['aquamaps_probability'].sum() > 0:
            y_prob = df_train['aquamaps_probability']
            print("🎯 Utilisation des probabilités AquaMaps comme target")
        else:
            y_prob = df_train['presence_probability']
            print("🎯 Utilisation des probabilités relatives comme target")
        
        # Convertir les probabilités en classes
        y = y_prob.apply(probability_to_class)
        
        print(f"\n📈 Distribution des classes dans les données d'entraînement :")
        class_counts = y.value_counts().sort_index()
        for class_id, count in class_counts.items():
            pct = count / len(y) * 100
            print(f"   Classe {class_id}: {count:3d} cellules ({pct:5.1f}%) - {class_to_probability_range(class_id)}")
        
        # Vérifier qu'on a au moins 2 classes
        if len(class_counts) < 2:
            print("⚠️ Une seule classe présente - impossible d'entraîner un modèle")
            ml_trained = False
        else:
            # Division train/test
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.3, random_state=42, stratify=y if len(class_counts) > 1 else None
            )
            
            print(f"\n🔄 Division des données :")
            print(f"   • Entraînement : {len(X_train)} cellules")
            print(f"   • Test : {len(X_test)} cellules")
            
            # Normalisation des features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Entraînement Random Forest
            print("\n🌲 Entraînement Random Forest...")
            rf_model = RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                random_state=42,
                class_weight='balanced'  # Pour gérer le déséquilibre des classes
            )
            
            rf_model.fit(X_train_scaled, y_train)
            
            # Prédictions
            y_pred_rf = rf_model.predict(X_test_scaled)
            accuracy_rf = accuracy_score(y_test, y_pred_rf)
            
            print(f"✅ Random Forest entraîné - Précision : {accuracy_rf:.3f}")
            
            # Entraînement Gradient Boosting
            print("\n⚡ Entraînement Gradient Boosting...")
            gb_model = GradientBoostingClassifier(
                n_estimators=100,
                max_depth=6,
                random_state=42
            )
            
            gb_model.fit(X_train_scaled, y_train)
            
            # Prédictions
            y_pred_gb = gb_model.predict(X_test_scaled)
            accuracy_gb = accuracy_score(y_test, y_pred_gb)
            
            print(f"✅ Gradient Boosting entraîné - Précision : {accuracy_gb:.3f}")
            
            # Sélectionner le meilleur modèle
            if accuracy_rf >= accuracy_gb:
                best_model = rf_model
                best_model_name = "Random Forest"
                best_accuracy = accuracy_rf
                y_pred_best = y_pred_rf
            else:
                best_model = gb_model
                best_model_name = "Gradient Boosting"
                best_accuracy = accuracy_gb
                y_pred_best = y_pred_gb
            
            print(f"\n🏆 Meilleur modèle : {best_model_name} (Précision : {best_accuracy:.3f})")
            
            # Rapport de classification
            print(f"\n📊 Rapport de classification ({best_model_name}) :")
            class_names = [class_to_probability_range(i) for i in range(5)]
            print(classification_report(y_test, y_pred_best, 
                                      target_names=class_names, 
                                      zero_division=0))
            
            ml_trained = True
            
else:
    print("❌ Entraînement impossible - données ou features manquantes")
    ml_trained = False

🤖 Entraînement du modèle de machine learning...
📊 Données d'entraînement : 1213 cellules avec observations
🎯 Utilisation des probabilités AquaMaps comme target

📈 Distribution des classes dans les données d'entraînement :
   Classe 0:  64 cellules (  5.3%) - 0.0-0.2 (Très improbable)
   Classe 1:  89 cellules (  7.3%) - 0.2-0.4 (Improbable)
   Classe 2: 105 cellules (  8.7%) - 0.4-0.6 (Possible)
   Classe 3: 191 cellules ( 15.7%) - 0.6-0.8 (Probable)
   Classe 4: 764 cellules ( 63.0%) - 0.8-1.0 (Très probable)

🔄 Division des données :
   • Entraînement : 849 cellules
   • Test : 364 cellules

🌲 Entraînement Random Forest...
✅ Random Forest entraîné - Précision : 0.821

⚡ Entraînement Gradient Boosting...
✅ Gradient Boosting entraîné - Précision : 0.799

🏆 Meilleur modèle : Random Forest (Précision : 0.821)

📊 Rapport de classification (Random Forest) :
                           precision    recall  f1-score   support

0.0-0.2 (Très improbable)       0.86      0.63      0.73        19

In [18]:
# Prédiction sur toutes les cellules et visualisation
if ml_trained:
    print("🔮 Prédiction sur toutes les cellules de la grille...")
    
    # Préparer les features pour toutes les cellules
    X_all = df_ml[feature_columns].copy()
    X_all_scaled = scaler.transform(X_all)
    
    # Prédictions
    y_pred_all = best_model.predict(X_all_scaled)
    y_pred_proba_all = best_model.predict_proba(X_all_scaled)
    
    # Ajouter les prédictions au DataFrame
    df_ml['predicted_class'] = y_pred_all
    df_ml['predicted_probability'] = y_pred_proba_all.max(axis=1)  # Probabilité de la classe prédite
    
    # Conversion de la classe en probabilité centrale pour visualisation
    def class_to_prob_center(class_pred):
        """Convertit une classe en probabilité centrale"""
        centers = {0: 0.1, 1: 0.3, 2: 0.5, 3: 0.7, 4: 0.9}
        return centers.get(class_pred, 0.5)
    
    df_ml['predicted_prob_center'] = df_ml['predicted_class'].apply(class_to_prob_center)
    
    print(f"✅ Prédictions effectuées sur {len(df_ml)} cellules")
    
    # Statistiques des prédictions
    print(f"\n📊 Distribution des prédictions sur toute la grille :")
    pred_counts = df_ml['predicted_class'].value_counts().sort_index()
    for class_id, count in pred_counts.items():
        pct = count / len(df_ml) * 100
        print(f"   Classe {class_id}: {count:4d} cellules ({pct:5.1f}%) - {class_to_probability_range(class_id)}")
    
    # Créer la carte de prédictions
    print(f"\n🗺️ Création de la carte de prédictions ML...")
    
    fig_ml = go.Figure()
    
    # Ajouter les prédictions
    fig_ml.add_trace(go.Scattermapbox(
        lat=df_ml['center_lat'],
        lon=df_ml['center_lon'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_ml['predicted_prob_center'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(
                title=dict(
                    text=f"Probabilité Prédite<br>({best_model_name})",
                    font=dict(size=12)
                ),
                thickness=15,
                len=0.7
            ),
            cmin=0,
            cmax=1,
            opacity=0.7
        ),
        text=[
            f"<b>Prédiction ML - {species_name.replace('_', ' ')}</b><br>" +
            f"🔢 Cellule: ({row['grid_lat']}, {row['grid_lon']})<br>" +
            f"📍 Centre: ({row['center_lat']:.2f}°, {row['center_lon']:.2f}°)<br>" +
            f"🤖 Classe prédite: {row['predicted_class']} - {class_to_probability_range(row['predicted_class'])}<br>" +
            f"📊 Confiance: {row['predicted_probability']:.3f}<br>" +
            f"🌡️ Temp estimée: {row['temperature_estimated']:.1f}°C<br>" +
            f"🌊 Prof estimée: {row['depth_estimated']:.0f}m<br>" +
            f"🧂 Salinité estimée: {row['salinity_estimated']:.1f}<br>" +
            f"🌍 Zone climatique: {['Polaire', 'Tempérée', 'Tropicale'][int(row['climate_zone'])]}"
            for _, row in df_ml.iterrows()
        ],
        hovertemplate='%{text}<extra></extra>',
        name=f'Prédictions ML'
    ))
    
    # Configuration de la carte
    fig_ml.update_layout(
        title=dict(
            text=f"🤖 Prédictions ML - Distribution de {species_name.replace('_', ' ')}<br>" +
                 f"<sub>Modèle: {best_model_name} | Précision: {best_accuracy:.3f} | 2,500 cellules prédites</sub>",
            x=0.5,
            font=dict(size=16)
        ),
        mapbox=dict(
            style="open-street-map",
            center=dict(lat=0, lon=0),
            zoom=1
        ),
        width=1200,
        height=700,
        margin=dict(l=0, r=0, t=100, b=0)
    )
    
    # Afficher la carte
    fig_ml.show()
    
    print(f"\n💡 INTERPRÉTATION DE LA CARTE ML :")
    print(f"   🎨 Couleur : Probabilité prédite (violet = faible, jaune = élevée)")
    print(f"   🖱️ Survol : Détails de la prédiction et features utilisées")
    print(f"   🤖 Modèle : {best_model_name} avec {best_accuracy:.1%} de précision")
    print(f"   🔮 Couverture : Prédictions sur les 2,500 cellules de la grille mondiale")
    
else:
    print("❌ Prédictions impossibles - modèle non entraîné")

🔮 Prédiction sur toutes les cellules de la grille...
✅ Prédictions effectuées sur 2500 cellules

📊 Distribution des prédictions sur toute la grille :
   Classe 0:  406 cellules ( 16.2%) - 0.0-0.2 (Très improbable)
   Classe 1:  538 cellules ( 21.5%) - 0.2-0.4 (Improbable)
   Classe 2:  448 cellules ( 17.9%) - 0.4-0.6 (Possible)
   Classe 3:  181 cellules (  7.2%) - 0.6-0.8 (Probable)
   Classe 4:  927 cellules ( 37.1%) - 0.8-1.0 (Très probable)

🗺️ Création de la carte de prédictions ML...



💡 INTERPRÉTATION DE LA CARTE ML :
   🎨 Couleur : Probabilité prédite (violet = faible, jaune = élevée)
   🖱️ Survol : Détails de la prédiction et features utilisées
   🤖 Modèle : Random Forest avec 82.1% de précision
   🔮 Couverture : Prédictions sur les 2,500 cellules de la grille mondiale


In [21]:
# Analyse de l'importance des features et comparaison avec AquaMaps
if ml_trained:
    print("📊 Analyse de l'importance des features...")
    
    # Importance des features
    feature_importance = best_model.feature_importances_
    feature_names = feature_columns
    
    # Créer un DataFrame pour l'importance
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print(f"\n🔍 Top 10 features les plus importantes ({best_model_name}) :")
    for i, (_, row) in enumerate(importance_df.head(10).iterrows(), 1):
        print(f"   {i:2d}. {row['feature']:25s} : {row['importance']:.4f}")
    
    # Comparaison avec AquaMaps si disponible
    if 'aquamaps_probability' in df_ml.columns and df_ml['aquamaps_probability'].sum() > 0:
        print(f"\n🆚 Comparaison ML vs AquaMaps...")
        
        # Filtrer les cellules avec données AquaMaps
        comparison_mask = df_ml['aquamaps_probability'] > 0
        df_compare = df_ml[comparison_mask].copy()
        
        if len(df_compare) > 0:
            # Corrélation entre prédictions ML et AquaMaps
            correlation = df_compare['predicted_prob_center'].corr(df_compare['aquamaps_probability'])
            
            print(f"   📈 Corrélation ML-AquaMaps : {correlation:.3f}")
            print(f"   📊 Cellules comparées : {len(df_compare)}")
            
            # Différences moyennes
            df_compare['diff_abs'] = abs(df_compare['predicted_prob_center'] - df_compare['aquamaps_probability'])
            mean_diff = df_compare['diff_abs'].mean()
            
            print(f"   📏 Différence absolue moyenne : {mean_diff:.3f}")
            
            # Zones de plus grande différence
            top_diff = df_compare.nlargest(5, 'diff_abs')
            print(f"\n   🎯 Top 5 des plus grandes différences :")
            for i, (_, row) in enumerate(top_diff.iterrows(), 1):
                print(f"{i}. Cellule ({int(row['grid_lat'])},{int(row['grid_lon'])}): ML={row['predicted_prob_center']:.2f}, AquaMaps={row['aquamaps_probability']:.2f}, Diff={row['diff_abs']:.2f}")
        
    # Zones d'intérêt identifiées par le ML
    high_prob_cells = df_ml[df_ml['predicted_class'] >= 3].copy()  # Probable ou très probable
    
    print(f"\n🎯 Zones d'intérêt identifiées par le ML :")
    print(f"   🔍 Cellules à forte probabilité (≥60%) : {len(high_prob_cells)}")
    
    if len(high_prob_cells) > 0:
        print(f"   🌍 Répartition géographique :")
        print(f"      • Latitude : {high_prob_cells['center_lat'].min():.1f}° à {high_prob_cells['center_lat'].max():.1f}°")
        print(f"      • Longitude : {high_prob_cells['center_lon'].min():.1f}° à {high_prob_cells['center_lon'].max():.1f}°")
        
        print(f"   🌡️ Conditions environnementales favorables :")
        print(f"      • Température : {high_prob_cells['temperature_estimated'].min():.1f}-{high_prob_cells['temperature_estimated'].max():.1f}°C")
        print(f"      • Profondeur : {high_prob_cells['depth_estimated'].min():.0f}-{high_prob_cells['depth_estimated'].max():.0f}m")
        print(f"      • Salinité : {high_prob_cells['salinity_estimated'].min():.1f}-{high_prob_cells['salinity_estimated'].max():.1f}")
    
    # Recommandations
    print(f"\n💡 RECOMMANDATIONS :")
    print(f"   🔬 Recherche : Explorer les zones à forte probabilité ML non documentées par AquaMaps")
    print(f"   🎣 Applications : Utiliser les prédictions pour planifier échantillonnage/conservation")
    print(f"   📈 Amélioration : Ajouter plus de données environnementales réelles")
    print(f"   🌍 Validation : Comparer avec observations terrain dans les zones prédites")
    
else:
    print("❌ Analyse impossible - modèle non entraîné")

📊 Analyse de l'importance des features...

🔍 Top 10 features les plus importantes (Random Forest) :
    1. lat_abs                   : 0.1175
    2. lat_south                 : 0.0883
    3. temperature_estimated     : 0.0881
    4. center_lat                : 0.0875
    5. lat_north                 : 0.0859
    6. lon_west                  : 0.0817
    7. lon_east                  : 0.0809
    8. center_lon                : 0.0808
    9. lon_abs                   : 0.0774
   10. salinity_estimated        : 0.0659

🆚 Comparaison ML vs AquaMaps...
   📈 Corrélation ML-AquaMaps : 0.939
   📊 Cellules comparées : 1213
   📏 Différence absolue moyenne : 0.074

   🎯 Top 5 des plus grandes différences :
1. Cellule (35,29): ML=0.90, AquaMaps=0.10, Diff=0.80
2. Cellule (24,17): ML=0.90, AquaMaps=0.22, Diff=0.68
3. Cellule (15,16): ML=0.70, AquaMaps=0.03, Diff=0.67
4. Cellule (25,17): ML=0.90, AquaMaps=0.29, Diff=0.61
5. Cellule (18,27): ML=0.90, AquaMaps=0.34, Diff=0.56

🎯 Zones d'intérêt identif

In [24]:
# CORRECTION: Mise à jour de l'assignation de grille pour les espèces individuelles
# Cette section corrige l'erreur "assign_to_grid() takes 2 positional arguments but 4 were given"

# Redéfinir la fonction d'assignation pour éviter les conflits
def assign_to_grid_fixed(lat, lon, grid_size=50):
    """Version fixe de la fonction d'assignation de grille"""
    return assign_to_grid_corrected(lat, lon, grid_size)

print("🔧 Fonction d'assignation de grille mise à jour pour les espèces individuelles")

# Analyse d'une espèce spécifique
# 🔧 CONFIGURATION - Modifiez cette ligne avec le nom de l'espèce souhaitée
species_name = "Acanthopagrus_schlegelii"  # ⬅️ CHANGEZ ICI pour une autre espèce

# S'assurer que tous les modules nécessaires sont importés
try:
    import pandas as pd
    import numpy as np
    import os
    import glob
except ImportError as e:
    print(f"❌ Erreur d'import: {e}")
    print("💡 Assurez-vous que pandas, numpy sont installés")

# Définir/redéfinir les fonctions de grille nécessaires
def grid_index(coord, min_val, max_val, grid_size):
    """Calculer l'index de grille pour une coordonnée"""
    if pd.isna(coord):
        return None
    
    # Normaliser la coordonnée entre 0 et 1
    normalized = (coord - min_val) / (max_val - min_val)
    
    # Calculer l'index de la grille (0 à grid_size-1)
    index = int(normalized * grid_size)
    index = max(0, min(index, grid_size - 1))
    
    return index

def get_cell_center_coords(grid_index, min_val, max_val, grid_size):
    """Obtenir le centre d'une cellule de grille (version 4 paramètres)"""
    if pd.isna(grid_index):
        return None
    cell_size = (max_val - min_val) / grid_size
    return min_val + (grid_index + 0.5) * cell_size

print("🔧 Fonction d'assignation de grille mise à jour pour les espèces individuelles")

# Définir grid_size si pas déjà défini
if 'grid_size' not in locals():
    grid_size = 50

# Paramètres d'affichage
show_details = True  # Afficher les détails de traitement
min_observations = 1  # Nombre minimum d'observations par cellule pour l'affichage

print(f"🎯 Analyse de l'espèce: {species_name}")
print(f"📁 Recherche du fichier: Complete CSVs/{species_name}.csv")

# Vérifier la disponibilité des espèces si la liste n'est pas déjà chargée
if 'available_species' not in locals():
    print("📂 Chargement de la liste des espèces disponibles...")
    csv_folder = "Complete CSVs"
    available_species = []
    
    if os.path.exists(csv_folder):
        csv_files = glob.glob(os.path.join(csv_folder, "*.csv"))
        available_species = [os.path.splitext(os.path.basename(f))[0] for f in csv_files]

# Vérifier si l'espèce est disponible
if available_species and species_name not in available_species:
    print(f"❌ Espèce '{species_name}' non trouvée")
    print(f"💡 Espèces disponibles: {len(available_species)} au total")
    print(f"🔍 Vérifiez l'orthographe ou choisissez parmi les espèces listées ci-dessus")
else:
    try:
        # Charger les données de l'espèce
        species_file = os.path.join("Complete CSVs", f"{species_name}.csv")
        
        if show_details:
            print(f"📊 Chargement des données...")
        
        # Lire le fichier CSV de l'espèce avec gestion du format spécial
        # Les fichiers d'espèces ont des métadonnées au début, les vraies données commencent vers la ligne 31
        try:
            # D'abord, trouver où commencent les vraies données (ligne avec "Genus,Species,...")
            with open(species_file, 'r', encoding='utf-8') as f:
                lines = f.readlines()
            
            header_line = None
            for i, line in enumerate(lines):
                if line.startswith('Genus,Species,'):
                    header_line = i
                    break
            
            if header_line is None:
                # Fallback: essayer de lire normalement avec gestion d'erreurs
                df_species = pd.read_csv(species_file, on_bad_lines='skip')
            else:
                # Lire à partir de la ligne d'en-tête trouvée avec gestion d'erreurs
                df_species = pd.read_csv(species_file, skiprows=header_line, on_bad_lines='skip')
                
        except UnicodeDecodeError:
            # Essayer avec un autre encodage
            try:
                with open(species_file, 'r', encoding='latin-1') as f:
                    lines = f.readlines()
                
                header_line = None
                for i, line in enumerate(lines):
                    if line.startswith('Genus,Species,'):
                        header_line = i
                        break
                
                if header_line is None:
                    df_species = pd.read_csv(species_file, encoding='latin-1', on_bad_lines='skip')
                else:
                    df_species = pd.read_csv(species_file, skiprows=header_line, encoding='latin-1', on_bad_lines='skip')
            except:
                # Dernier recours avec paramètres très permissifs
                df_species = pd.read_csv(species_file, sep=',', on_bad_lines='skip', quoting=1, error_bad_lines=False)
        
        print(f"✅ Données chargées: {len(df_species):,} observations")
        
        # Afficher les premières lignes pour comprendre la structure
        if show_details:
            print(f"\n📋 Structure des données:")
            print(f"   Colonnes: {list(df_species.columns)}")
            print(f"   Premières lignes:")
            print(df_species.head(3))
        
        # Vérifier les colonnes nécessaires et adapter aux noms utilisés dans les fichiers d'espèces
        # Les fichiers d'espèces utilisent "Center Lat" et "Center Long"
        if 'Center Lat' in df_species.columns and 'Center Long' in df_species.columns:
            # Renommer pour correspondre au code existant  
            df_species = df_species.rename(columns={
                'Center Lat': 'center_lat',
                'Center Long': 'center_long'
            })
            required_cols = ['center_lat', 'center_long']
            missing_cols = []
        else:
            required_cols = ['center_lat', 'center_long']
            missing_cols = [col for col in required_cols if col not in df_species.columns]
        
        if missing_cols:
            print(f"❌ Colonnes manquantes: {missing_cols}")
            print(f"💡 Colonnes disponibles: {list(df_species.columns)}")
        else:
            # Nettoyer les données
            df_species_clean = df_species.dropna(subset=['center_lat', 'center_long'])
            
            print(f"🧹 Données nettoyées: {len(df_species_clean):,} observations avec coordonnées valides")
            
            if len(df_species_clean) == 0:
                print(f"❌ Aucune observation avec coordonnées valides")
            else:
                # Assigner les observations aux cellules de la grille
                if show_details:
                    print(f"🗺️ Attribution aux cellules de la grille...")
                
                df_species_clean['grid_lat'] = df_species_clean['center_lat'].apply(
                    lambda lat: grid_index(lat, -90, 90, grid_size)
                )
                df_species_clean['grid_lon'] = df_species_clean['center_long'].apply(
                    lambda lon: grid_index(lon, -180, 180, grid_size)
                )
                
                # Créer l'identifiant de cellule
                df_species_clean['cell_id'] = df_species_clean['grid_lat'].astype(str) + '_' + df_species_clean['grid_lon'].astype(str)
                
                # Compter les observations par cellule et calculer les probabilités moyennes
                agg_dict = {
                    'center_lat': ['mean', 'count'],
                    'center_long': 'mean'
                }
                
                # Ajouter la probabilité globale si elle existe
                if 'Overall Probability' in df_species_clean.columns:
                    agg_dict['Overall Probability'] = ['mean', 'std']
                
                species_grid = df_species_clean.groupby(['grid_lat', 'grid_lon', 'cell_id']).agg(agg_dict).reset_index()
                
                # Simplifier les noms de colonnes
                if 'Overall Probability' in df_species_clean.columns:
                    species_grid.columns = ['grid_lat', 'grid_lon', 'cell_id', 'mean_lat', 'observation_count', 'mean_lon', 'mean_probability', 'std_probability']
                else:
                    species_grid.columns = ['grid_lat', 'grid_lon', 'cell_id', 'mean_lat', 'observation_count', 'mean_lon']
                
                # Calculer les centres des cellules
                species_grid['cell_center_lat'] = species_grid['grid_lat'].apply(
                    lambda i: get_cell_center_coords(i, -90, 90, grid_size)
                )
                species_grid['cell_center_lon'] = species_grid['grid_lon'].apply(
                    lambda i: get_cell_center_coords(i, -180, 180, grid_size)
                )
                
                # Calculer la probabilité de présence
                if 'mean_probability' in species_grid.columns:
                    # Utiliser les vraies probabilités moyennes d'AquaMaps
                    species_grid['presence_probability'] = species_grid['mean_probability']
                else:
                    # Fallback: utiliser le nombre d'observations relatif
                    max_obs = species_grid['observation_count'].max()
                    species_grid['presence_probability'] = species_grid['observation_count'] / max_obs
                
                # Filtrer par nombre minimum d'observations
                species_grid_filtered = species_grid[species_grid['observation_count'] >= min_observations]
                
                print(f"📊 Résultats de l'agrégation:")
                print(f"   🔢 Cellules avec données: {len(species_grid)}")
                print(f"   📍 Cellules affichées (≥{min_observations} obs): {len(species_grid_filtered)}")
                print(f"   📈 Observations par cellule: min={species_grid['observation_count'].min()}, max={species_grid['observation_count'].max()}")
                
                # Afficher les statistiques de probabilité si disponibles
                if 'mean_probability' in species_grid.columns:
                    print(f"   🎯 Probabilité AquaMaps: min={species_grid['mean_probability'].min():.3f}, max={species_grid['mean_probability'].max():.3f}")
                    print(f"   📊 Probabilité moyenne: {species_grid['mean_probability'].mean():.3f} ± {species_grid['mean_probability'].std():.3f}")
                
                print(f"   🌍 Couverture géographique:")
                print(f"      Latitude: {species_grid['mean_lat'].min():.2f}° à {species_grid['mean_lat'].max():.2f}°")
                print(f"      Longitude: {species_grid['mean_lon'].min():.2f}° à {species_grid['mean_lon'].max():.2f}°")
                
    except FileNotFoundError:
        print(f"❌ Fichier non trouvé: {species_file}")
        print(f"💡 Vérifiez que le fichier existe dans le dossier 'Complete CSVs'")
    except Exception as e:
        print(f"❌ Erreur lors du chargement: {e}")
        print(f"💡 Vérifiez le format du fichier CSV")

🔧 Fonction d'assignation de grille mise à jour pour les espèces individuelles
🔧 Fonction d'assignation de grille mise à jour pour les espèces individuelles
🎯 Analyse de l'espèce: Acanthopagrus_schlegelii
📁 Recherche du fichier: Complete CSVs/Acanthopagrus_schlegelii.csv
📊 Chargement des données...
✅ Données chargées: 467 observations

📋 Structure des données:
   Colonnes: ['Genus', 'Species', 'Center Lat', 'Center Long', 'C-Square Code', 'Overall Probability']
   Premières lignes:
           Genus     Species  Center Lat  Center Long C-Square Code  \
0  Acanthopagrus  schlegelii       41.25       139.75    1413:219:2   
1  Acanthopagrus  schlegelii       37.25       131.75    1313:371:2   
2  Acanthopagrus  schlegelii       37.75       130.75    1313:370:4   

   Overall Probability  
0                 0.05  
1                 0.04  
2                 0.08  
🧹 Données nettoyées: 466 observations avec coordonnées valides
🗺️ Attribution aux cellules de la grille...
📊 Résultats de l'agrég

In [ ]:
# Fonction auxiliaire pour calculer l'index de la grille
def grid_index(coord, min_val, max_val, grid_size):
    """Calculer l'index de la grille pour une coordonnée"""
    grid_idx = int((coord - min_val) / ((max_val - min_val) / grid_size))
    return max(0, min(grid_idx, grid_size - 1))

print("✅ Fonction grid_index définie avec succès")

✅ Fonction grid_index définie avec succès


# 🔬 Évaluation Robuste par Cross-Validation

Maintenant que nous avons testé notre modèle sur un simple split train/test, nous allons effectuer une **évaluation plus rigoureuse** avec la **cross-validation 5-fold**.

## 🎯 Objectifs :
- **Validation robuste** : Tester le modèle sur 5 échantillons différents
- **Métriques détaillées** : Analyser les performances par classe
- **Stabilité** : Mesurer la cohérence des prédictions
- **Matrice de confusion** : Identifier les erreurs de classification

## 📊 Processus :
1. **5-Fold Stratified CV** : Maintient la distribution des classes
2. **Métriques par fold** : Accuracy, Precision, Recall, F1-Score
3. **Analyse statistique** : Moyenne, écart-type, intervalles de confiance
4. **Visualisations** : Graphiques des performances et matrices de confusion

In [ ]:
# 🔄 Préparation des données pour Cross-Validation
print("🔄 Préparation de la Cross-Validation 5-Fold...")

# Vérifier que nous avons les données d'entraînement
if 'X_train' not in locals() or 'y_train' not in locals():
    print("❌ Erreur: Les données d'entraînement ne sont pas disponibles")
    print("💡 Veuillez d'abord exécuter les cellules de machine learning précédentes")
else:
    print(f"✅ Données disponibles:")
    print(f"   • Features: {X_train.shape}")
    print(f"   • Classes: {len(y_train)}")
    
    # Préparer les données pour la cross-validation (utiliser toutes les données)
    # Recombiner X_train et X_test pour la cross-validation complète
    X_cv = pd.concat([X_train, X_test], axis=0, ignore_index=True)
    y_cv = pd.concat([y_train, y_test], axis=0, ignore_index=True)
    
    print(f"   • Données CV complètes: {X_cv.shape}")
    
    # Vérifier la distribution des classes
    from collections import Counter
    class_distribution = Counter(y_cv)
    print(f"\n📊 Distribution des classes:")
    for class_id in sorted(class_distribution.keys()):
        count = class_distribution[class_id]
        percentage = (count / len(y_cv)) * 100
        class_name = ['Très improbable', 'Improbable', 'Possible', 'Probable', 'Très probable'][class_id]
        print(f"   • Classe {class_id} ({class_name}): {count:,} observations ({percentage:.1f}%)")
    
    # Imports nécessaires
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    print(f"\n🧰 Modules importés avec succès")
    print(f"🎯 Prêt pour la cross-validation !")

🔄 Préparation de la Cross-Validation 5-Fold...
❌ Erreur: Les données d'entraînement ne sont pas disponibles
💡 Veuillez d'abord exécuter les cellules de machine learning précédentes


In [ ]:
# 🌲 Cross-Validation avec Random Forest
print("🌲 Évaluation Random Forest avec 5-Fold Cross-Validation...")

# Configuration du modèle Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Configuration de la cross-validation stratifiée
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Stockage des résultats
fold_results = {
    'accuracy': [],
    'precision_macro': [],
    'recall_macro': [],
    'f1_macro': [],
    'confusion_matrices': [],
    'classification_reports': []
}

print(f"🔄 Début de la cross-validation...")

# Cross-validation manuelle pour obtenir des métriques détaillées
for fold, (train_idx, val_idx) in enumerate(cv_strategy.split(X_cv, y_cv)):
    print(f"\n📊 Fold {fold + 1}/5:")
    
    # Division des données
    X_fold_train, X_fold_val = X_cv.iloc[train_idx], X_cv.iloc[val_idx]
    y_fold_train, y_fold_val = y_cv.iloc[train_idx], y_cv.iloc[val_idx]
    
    print(f"   • Train: {len(X_fold_train):,} observations")
    print(f"   • Test:  {len(X_fold_val):,} observations")
    
    # Entraînement du modèle
    rf_model.fit(X_fold_train, y_fold_train)
    
    # Prédictions
    y_pred = rf_model.predict(X_fold_val)
    
    # Calcul des métriques
    accuracy = accuracy_score(y_fold_val, y_pred)
    
    # Rapport de classification détaillé
    report = classification_report(y_fold_val, y_pred, output_dict=True, zero_division=0)
    
    # Matrice de confusion
    cm = confusion_matrix(y_fold_val, y_pred)
    
    # Stockage des résultats
    fold_results['accuracy'].append(accuracy)
    fold_results['precision_macro'].append(report['macro avg']['precision'])
    fold_results['recall_macro'].append(report['macro avg']['recall'])
    fold_results['f1_macro'].append(report['macro avg']['f1-score'])
    fold_results['confusion_matrices'].append(cm)
    fold_results['classification_reports'].append(report)
    
    print(f"   ✅ Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
    print(f"   📊 F1-Score macro: {report['macro avg']['f1-score']:.3f}")

print(f"\n🎯 Cross-validation terminée !")

# Calcul des statistiques finales
print(f"\n📈 RÉSULTATS FINAUX - Random Forest:")
print(f"{'='*50}")

metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
metric_names = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1-Score (macro)']

for metric, name in zip(metrics, metric_names):
    values = fold_results[metric]
    mean_val = np.mean(values)
    std_val = np.std(values)
    
    print(f"{name:20}: {mean_val:.3f} ± {std_val:.3f} ({mean_val*100:.1f}% ± {std_val*100:.1f}%)")

print(f"\n🔍 Détail par fold:")
for i in range(5):
    acc = fold_results['accuracy'][i]
    f1 = fold_results['f1_macro'][i]
    print(f"   Fold {i+1}: Accuracy={acc:.3f} ({acc*100:.1f}%), F1={f1:.3f}")

In [ ]:
# 📊 Analyse Détaillée par Classe
print("📊 Analyse des performances par classe de probabilité...")

# Calcul des métriques moyennes par classe
class_names = ['Très improbable', 'Improbable', 'Possible', 'Probable', 'Très probable']
n_classes = len(class_names)

# Agrégation des rapports de classification
class_metrics = {
    'precision': [[] for _ in range(n_classes)],
    'recall': [[] for _ in range(n_classes)],
    'f1-score': [[] for _ in range(n_classes)],
    'support': [[] for _ in range(n_classes)]
}

for report in fold_results['classification_reports']:
    for class_id in range(n_classes):
        if str(class_id) in report:  # Vérifier que la classe existe dans ce fold
            class_metrics['precision'][class_id].append(report[str(class_id)]['precision'])
            class_metrics['recall'][class_id].append(report[str(class_id)]['recall'])
            class_metrics['f1-score'][class_id].append(report[str(class_id)]['f1-score'])
            class_metrics['support'][class_id].append(report[str(class_id)]['support'])

# Affichage des résultats par classe
print(f"\n🎯 PERFORMANCES PAR CLASSE:")
print(f"{'='*80}")
print(f"{'Classe':<15} {'Nom':<18} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print(f"{'-'*80}")

for class_id in range(n_classes):
    if class_metrics['precision'][class_id]:  # Si la classe a des données
        prec_mean = np.mean(class_metrics['precision'][class_id])
        prec_std = np.std(class_metrics['precision'][class_id])
        
        rec_mean = np.mean(class_metrics['recall'][class_id])
        rec_std = np.std(class_metrics['recall'][class_id])
        
        f1_mean = np.mean(class_metrics['f1-score'][class_id])
        f1_std = np.std(class_metrics['f1-score'][class_id])
        
        support_mean = np.mean(class_metrics['support'][class_id])
        
        print(f"Classe {class_id:<8} {class_names[class_id]:<18} "
              f"{prec_mean:.3f}±{prec_std:.3f} {rec_mean:.3f}±{rec_std:.3f} "
              f"{f1_mean:.3f}±{f1_std:.3f} {support_mean:.0f}")

# Analyse des erreurs les plus fréquentes
print(f"\n🔍 ANALYSE DES ERREURS:")
print(f"{'='*50}")

# Moyenne des matrices de confusion
mean_confusion_matrix = np.mean(fold_results['confusion_matrices'], axis=0)

print(f"📈 Matrice de confusion moyenne (sur 5 folds):")
print(f"Lignes = Vraies classes, Colonnes = Prédictions")
print(f"\n     ", end="")
for i in range(n_classes):
    print(f"Pred {i:>6}", end="")
print()

for i in range(n_classes):
    print(f"Vrai {i}: ", end="")
    for j in range(n_classes):
        print(f"{mean_confusion_matrix[i, j]:>7.1f}", end="")
    print(f"  ({class_names[i]})")

# Calcul des erreurs les plus fréquentes
print(f"\n❌ Erreurs de classification les plus fréquentes:")
for i in range(n_classes):
    for j in range(n_classes):
        if i != j and mean_confusion_matrix[i, j] > 5:  # Seuil d'erreurs significatives
            error_rate = mean_confusion_matrix[i, j] / np.sum(mean_confusion_matrix[i, :]) * 100
            print(f"   • {class_names[i]} → {class_names[j]}: "
                  f"{mean_confusion_matrix[i, j]:.1f} erreurs ({error_rate:.1f}%)")

# Classes les mieux/moins bien prédites
print(f"\n🏆 CLASSEMENT DES PERFORMANCES:")
class_f1_scores = []
for class_id in range(n_classes):
    if class_metrics['f1-score'][class_id]:
        f1_mean = np.mean(class_metrics['f1-score'][class_id])
        class_f1_scores.append((class_id, f1_mean, class_names[class_id]))

class_f1_scores.sort(key=lambda x: x[1], reverse=True)

print(f"📊 Classes par ordre de performance (F1-Score):")
for rank, (class_id, f1_score, name) in enumerate(class_f1_scores, 1):
    emoji = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "📍"
    print(f"   {emoji} {rank}. Classe {class_id} ({name}): F1 = {f1_score:.3f} ({f1_score*100:.1f}%)")